# External OD/OC segmenter development

**Public role:** Frozen segmenter-development notebook.

**Relation to the manuscript:** Supports independent OD/OC proxy development. Methodological content is preserved.

**Frozen methodology:** The public curation changes only documentation, path abstraction, notebook metadata, and stored outputs. The experimental method and parameters are unchanged.

**Execution status of this curated copy:** Stored outputs and execution counters were removed to prevent disclosure of local paths, embedded dataset images, stale errors, and large widget state. The code was not re-executed after this mechanical curation. Reference outputs are distributed separately under `results/`.

**Path configuration:** Set `AQPR_PROJECT_ROOT` to the working directory containing `project_data/`, `models/`, and `results/`. Set `AQPR_DATA_ROOT` to the external dataset root. The default values target a Google Drive/Colab workflow and can be changed without editing the notebook logic.

**Original source notebook:** `04_external_od_oc_segmentor_training_and_validation.ipynb`


In [ ]:
import os
# Cell 1: Load paths and register external OD/OC datasets

from pathlib import Path
import json

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

JOURNAL_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))
CONFIG_PATH = JOURNAL_ROOT / "configs" / "project_config.json"
PATH_REGISTRY_PATH = JOURNAL_ROOT / "configs" / "path_registry.csv"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Missing config: {CONFIG_PATH}")

if not PATH_REGISTRY_PATH.exists():
    raise FileNotFoundError(f"Missing path registry: {PATH_REGISTRY_PATH}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)

path_registry = pd.read_csv(PATH_REGISTRY_PATH)
PATHS = {
    row["path_key"]: Path(row["path"])
    for _, row in path_registry.iterrows()
}

LEGACY_ROOT = Path(PROJECT_CONFIG["legacy_root"])
EXTERNAL_ROOT = PATHS["external_datasets"]

expected_datasets = {
    "REFUGE": {
        "task_role": "od_oc_segmentation_and_cross_dataset_validation",
        "primary_training_candidate": True,
    },
    "G1020": {
        "task_role": "od_oc_segmentation_and_cross_dataset_validation",
        "primary_training_candidate": True,
    },
    "RIGA": {
        "task_role": "od_oc_segmentation_and_cross_dataset_validation",
        "primary_training_candidate": True,
    },
    "PAPILA": {
        "task_role": "od_oc_segmentation_and_cross_dataset_validation",
        "primary_training_candidate": True,
    },
}

for dataset_name in expected_datasets:
    (EXTERNAL_ROOT / dataset_name).mkdir(parents=True, exist_ok=True)

search_roots = [
    EXTERNAL_ROOT,
    LEGACY_ROOT / "dataset",
    LEGACY_ROOT,
]

dataset_rows = []

for dataset_name, dataset_info in expected_datasets.items():
    candidate_dirs = []

    for root in search_roots:
        if not root.exists():
            continue

        exact_path = root / dataset_name
        if exact_path.exists() and exact_path.is_dir():
            candidate_dirs.append(exact_path)

        keyword_matches = [
            p for p in root.rglob(f"*{dataset_name}*")
            if p.is_dir()
        ]

        candidate_dirs.extend(keyword_matches)

    candidate_dirs = sorted(set(candidate_dirs), key=lambda p: len(str(p)))

    selected_path = candidate_dirs[0] if candidate_dirs else EXTERNAL_ROOT / dataset_name
    selected_exists = selected_path.exists()
    n_files = sum(1 for p in selected_path.rglob("*") if p.is_file()) if selected_exists else 0

    dataset_rows.append(
        {
            "dataset": dataset_name,
            "selected_path": str(selected_path),
            "exists": selected_exists,
            "n_files": int(n_files),
            "task_role": dataset_info["task_role"],
            "primary_training_candidate": dataset_info["primary_training_candidate"],
            "status": "available" if selected_exists and n_files > 0 else "empty_or_missing",
        }
    )

od_dataset_registry = pd.DataFrame(dataset_rows)

legacy_segmentor_reference = pd.DataFrame(
    [
        {
            "artifact": "legacy_od_segmentor_weights",
            "path": str(LEGACY_ROOT / "models/optic_disc_segmentor/unet_refuge_od_512_seed42/refuge_unet_od_segmentor_best_weights_only.pt"),
            "exists": (LEGACY_ROOT / "models/optic_disc_segmentor/unet_refuge_od_512_seed42/refuge_unet_od_segmentor_best_weights_only.pt").exists(),
            "usage": "legacy_reference_only",
        },
        {
            "artifact": "legacy_od_segmentor_validation_metrics",
            "path": str(LEGACY_ROOT / "results/optic_disc_segmentor/unet_refuge_od_512_seed42/refuge_unet_od_segmentor_final_anatomical_validation_per_sample_metrics.csv"),
            "exists": (LEGACY_ROOT / "results/optic_disc_segmentor/unet_refuge_od_512_seed42/refuge_unet_od_segmentor_final_anatomical_validation_per_sample_metrics.csv").exists(),
            "usage": "legacy_reference_only",
        },
    ]
)

dataset_registry_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_external_od_oc_dataset_registry.csv"
legacy_reference_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_legacy_segmentor_reference.csv"

od_dataset_registry.to_csv(dataset_registry_csv, index=False)
legacy_segmentor_reference.to_csv(legacy_reference_csv, index=False)

available_count = int((od_dataset_registry["status"] == "available").sum())

print(f"External datasets available: {available_count}/{len(expected_datasets)}")
print(od_dataset_registry[["dataset", "exists", "n_files", "status"]].to_string(index=False))
print(f"Dataset registry: {dataset_registry_csv}")

if available_count == 0:
    raise FileNotFoundError(f"No external OD/OC datasets found under: {EXTERNAL_ROOT}")

print("Status: PASS")

In [ ]:
# Cell 2: Define external OD/OC dataset specifications

dataset_specs = {
    "REFUGE": {
        "annotation_format": "mask_images",
        "segmentation_targets": "OD_OC",
        "diagnosis_labels": True,
        "patient_ids": False,
        "expected_images": 1200,
        "expected_annotation_unit": "mask_per_image",
        "manifest_strategy": "match_images_to_mask_files",
    },
    "G1020": {
        "annotation_format": "labelme_json",
        "segmentation_targets": "OD_OC_bbox",
        "diagnosis_labels": True,
        "patient_ids": True,
        "expected_images": 1020,
        "expected_annotation_unit": "json_per_image",
        "manifest_strategy": "parse_labelme_json_annotations",
    },
    "RIGA": {
        "annotation_format": "expert_manual_markings",
        "segmentation_targets": "OD_OC",
        "diagnosis_labels": False,
        "patient_ids": False,
        "expected_images": 750,
        "expected_annotation_unit": "six_expert_markings_per_image",
        "manifest_strategy": "extract_original_and_expert_marked_pairs",
    },
    "PAPILA": {
        "annotation_format": "expert_segmentations",
        "segmentation_targets": "OD_OC",
        "diagnosis_labels": True,
        "patient_ids": True,
        "expected_images": 488,
        "expected_annotation_unit": "expert_contours_or_masks",
        "manifest_strategy": "match_fundus_images_to_expert_segmentations",
    },
}

spec_rows = []

for _, row in od_dataset_registry.iterrows():
    dataset = row["dataset"]
    root = Path(row["selected_path"])
    spec = dataset_specs[dataset]

    n_files = sum(1 for p in root.rglob("*") if p.is_file()) if root.exists() else 0

    spec_rows.append(
        {
            "dataset": dataset,
            "root": str(root),
            "root_exists": root.exists(),
            "n_files": int(n_files),
            **spec,
        }
    )

od_oc_dataset_specs = pd.DataFrame(spec_rows).sort_values("dataset").reset_index(drop=True)

specs_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_od_oc_dataset_specs.csv"
od_oc_dataset_specs.to_csv(specs_csv, index=False)

available_datasets = od_oc_dataset_specs.loc[
    od_oc_dataset_specs["root_exists"] & (od_oc_dataset_specs["n_files"] > 0),
    "dataset",
].tolist()

print(f"Available datasets: {available_datasets}")
print(od_oc_dataset_specs[["dataset", "annotation_format", "expected_images", "n_files", "manifest_strategy"]].to_string(index=False))
print(f"Dataset specs: {specs_csv}")

if set(available_datasets) != set(dataset_specs.keys()):
    missing = sorted(set(dataset_specs.keys()) - set(available_datasets))
    raise RuntimeError(f"Missing datasets: {missing}")

print("Status: PASS")

In [ ]:
# Cell 3: Audit dataset-specific segmentation resources

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}
ANNOTATION_EXTENSIONS = {".json", ".mat", ".csv", ".txt", ".xml"}

def count_files(root, extensions=None):
    files = [p for p in root.rglob("*") if p.is_file()]
    if extensions is None:
        return len(files)
    return sum(1 for p in files if p.suffix.lower() in extensions)

def count_in_subdir(root, subdir, extensions=None):
    path = root / subdir
    if not path.exists():
        return 0
    return count_files(path, extensions)

audit_rows = []

for _, row in od_oc_dataset_specs.iterrows():
    dataset = row["dataset"]
    root = Path(row["root"])

    if dataset == "REFUGE":
        n_images = (
            count_in_subdir(root, "train/images", IMAGE_EXTENSIONS) +
            count_in_subdir(root, "val/images", IMAGE_EXTENSIONS) +
            count_in_subdir(root, "test/images", IMAGE_EXTENSIONS)
        )
        n_annotations = (
            count_in_subdir(root, "train/mask", IMAGE_EXTENSIONS) +
            count_in_subdir(root, "val/mask", IMAGE_EXTENSIONS) +
            count_in_subdir(root, "test/mask", IMAGE_EXTENSIONS)
        )
        ready = n_images == 1200 and n_annotations == 1200

    elif dataset == "G1020":
        n_images = count_files(root, IMAGE_EXTENSIONS)
        n_annotations = count_files(root, {".json"})
        ready = n_images == 1020 and n_annotations >= 1020

    elif dataset == "PAPILA":
        n_images = count_in_subdir(root, "FundusImages", IMAGE_EXTENSIONS)
        n_annotations = count_in_subdir(root, "ExpertsSegmentations", None)
        ready = n_images > 0 and n_annotations > 0

    elif dataset == "RIGA":
        n_images = count_files(root, IMAGE_EXTENSIONS)
        n_annotations = n_images - 750 if n_images >= 750 else 0
        ready = n_images >= 5250 and n_annotations >= 4500

    else:
        n_images = count_files(root, IMAGE_EXTENSIONS)
        n_annotations = count_files(root, ANNOTATION_EXTENSIONS)
        ready = False

    audit_rows.append(
        {
            "dataset": dataset,
            "root": str(root),
            "annotation_format": row["annotation_format"],
            "n_image_files": int(n_images),
            "n_annotation_files_or_markings": int(n_annotations),
            "segmentation_ready": bool(ready),
        }
    )

segmentation_resource_audit = pd.DataFrame(audit_rows).sort_values("dataset").reset_index(drop=True)

resource_audit_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_segmentation_resource_audit.csv"
segmentation_resource_audit.to_csv(resource_audit_csv, index=False)

ready_count = int(segmentation_resource_audit["segmentation_ready"].sum())

print(f"Segmentation-ready datasets: {ready_count}/{len(segmentation_resource_audit)}")
print(
    segmentation_resource_audit[
        ["dataset", "annotation_format", "n_image_files", "n_annotation_files_or_markings", "segmentation_ready"]
    ].to_string(index=False)
)
print(f"Resource audit: {resource_audit_csv}")

if ready_count == 0:
    raise RuntimeError("No dataset is ready for OD/OC segmentation training.")

print("Status: PASS")

In [ ]:
# Cell 4: Define OD/OC segmentation training plan

training_plan = segmentation_resource_audit.copy()

training_plan["segmentation_use"] = np.where(
    training_plan["segmentation_ready"],
    "train_and_cross_validate",
    "pending_annotations",
)

training_plan["reason"] = ""

training_plan.loc[
    training_plan["dataset"] == "G1020",
    "reason"
] = "Local copy has images but no LabelMe JSON annotations."

training_plan.loc[
    training_plan["dataset"] == "RIGA",
    "reason"
] = "Use for OD/OC segmentation only; no glaucoma diagnosis labels."

training_plan.loc[
    training_plan["dataset"] == "REFUGE",
    "reason"
] = "Use image-mask pairs for OD/OC segmentation training."

training_plan.loc[
    training_plan["dataset"] == "PAPILA",
    "reason"
] = "Use fundus images and expert segmentations for OD/OC segmentation training."

training_plan["primary_segmentation_training"] = (
    training_plan["segmentation_use"] == "train_and_cross_validate"
)

training_plan["diagnostic_external_validation_candidate"] = training_plan["dataset"].isin(
    ["G1020", "PAPILA", "REFUGE"]
)

training_plan["anatomical_external_validation_candidate"] = training_plan["dataset"].isin(
    ["REFUGE", "PAPILA", "RIGA"]
)

training_plan = training_plan[
    [
        "dataset",
        "annotation_format",
        "segmentation_use",
        "primary_segmentation_training",
        "anatomical_external_validation_candidate",
        "diagnostic_external_validation_candidate",
        "n_image_files",
        "n_annotation_files_or_markings",
        "reason",
        "root",
    ]
].sort_values("dataset").reset_index(drop=True)

training_plan_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_segmentation_training_plan.csv"
training_plan.to_csv(training_plan_csv, index=False)

trainable_datasets = training_plan.loc[
    training_plan["primary_segmentation_training"], "dataset"
].tolist()

pending_datasets = training_plan.loc[
    training_plan["segmentation_use"] == "pending_annotations", "dataset"
].tolist()

print(f"Trainable segmentation datasets: {trainable_datasets}")
print(f"Pending segmentation datasets: {pending_datasets}")
print(f"Training plan: {training_plan_csv}")

if len(trainable_datasets) < 2:
    raise RuntimeError("At least two trainable datasets are required for cross-dataset validation.")

print("Status: PASS")

In [ ]:
# Cell 5: Build REFUGE image-mask manifest

REFUGE_ROOT = Path(
    training_plan.loc[training_plan["dataset"] == "REFUGE", "root"].iloc[0]
)

refuge_rows = []

for split in ["train", "val", "test"]:
    image_dir = REFUGE_ROOT / split / "images"
    mask_dir = REFUGE_ROOT / split / "mask"

    if not image_dir.exists():
        raise FileNotFoundError(f"Missing REFUGE image directory: {image_dir}")

    if not mask_dir.exists():
        raise FileNotFoundError(f"Missing REFUGE mask directory: {mask_dir}")

    image_files = sorted(
        [p for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
    )
    mask_files = sorted(
        [p for p in mask_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
    )

    mask_by_stem = {p.stem: p for p in mask_files}
    missing_masks = []

    for image_path in image_files:
        mask_path = mask_by_stem.get(image_path.stem)

        if mask_path is None:
            missing_masks.append(image_path.name)
            continue

        refuge_rows.append(
            {
                "dataset": "REFUGE",
                "split_original": split,
                "sample_id": f"REFUGE_{split}_{image_path.stem}",
                "image_name": image_path.name,
                "mask_name": mask_path.name,
                "image_path": str(image_path),
                "mask_path": str(mask_path),
                "image_exists": image_path.exists(),
                "mask_exists": mask_path.exists(),
            }
        )

    if missing_masks:
        raise FileNotFoundError(f"Missing REFUGE masks in split {split}: {missing_masks[:20]}")

refuge_manifest = pd.DataFrame(refuge_rows)

refuge_manifest_dir = PATHS["project_data"] / "external_datasets" / "REFUGE"
refuge_manifest_dir.mkdir(parents=True, exist_ok=True)

refuge_manifest_csv = refuge_manifest_dir / "refuge_od_oc_manifest.csv"
refuge_manifest_results_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_refuge_od_oc_manifest.csv"

refuge_manifest.to_csv(refuge_manifest_csv, index=False)
refuge_manifest.to_csv(refuge_manifest_results_csv, index=False)

split_counts = refuge_manifest["split_original"].value_counts().to_dict()
missing_files = int((~refuge_manifest["image_exists"]).sum() + (~refuge_manifest["mask_exists"]).sum())

print(f"REFUGE samples: {len(refuge_manifest)}")
print(f"REFUGE split counts: {split_counts}")
print(f"Missing files: {missing_files}")
print(f"REFUGE manifest: {refuge_manifest_csv}")

if len(refuge_manifest) != 1200:
    raise RuntimeError(f"Expected 1200 REFUGE samples, found {len(refuge_manifest)}")

if missing_files > 0:
    raise RuntimeError("Missing REFUGE image or mask files found.")

print("Status: PASS")

In [ ]:
# Cell 6: REFUGE sample image-mask audit

import cv2

rng = np.random.default_rng(int(PROJECT_CONFIG["seed"]))
sample_rows = []

for split in ["train", "val", "test"]:
    split_df = refuge_manifest[refuge_manifest["split_original"] == split].copy()
    sample_size = min(8, len(split_df))
    sample_indices = rng.choice(split_df.index.to_numpy(), size=sample_size, replace=False)

    for idx in sample_indices:
        row = refuge_manifest.loc[idx]

        image = cv2.imread(row["image_path"], cv2.IMREAD_COLOR)
        mask = cv2.imread(row["mask_path"], cv2.IMREAD_UNCHANGED)

        if image is None:
            raise RuntimeError(f"Unreadable REFUGE image: {row['image_path']}")

        if mask is None:
            raise RuntimeError(f"Unreadable REFUGE mask: {row['mask_path']}")

        if mask.ndim == 3:
            mask_gray = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
        else:
            mask_gray = mask

        unique_values = sorted(np.unique(mask_gray).astype(int).tolist())

        sample_rows.append(
            {
                "dataset": "REFUGE",
                "split_original": split,
                "sample_id": row["sample_id"],
                "image_height": int(image.shape[0]),
                "image_width": int(image.shape[1]),
                "image_channels": int(image.shape[2]),
                "mask_height": int(mask_gray.shape[0]),
                "mask_width": int(mask_gray.shape[1]),
                "mask_unique_values": "|".join(map(str, unique_values)),
            }
        )

refuge_sample_audit = pd.DataFrame(sample_rows)

refuge_sample_audit_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_refuge_sample_audit.csv"
refuge_sample_audit.to_csv(refuge_sample_audit_csv, index=False)

shape_mismatch = int(
    (
        (refuge_sample_audit["image_height"] != refuge_sample_audit["mask_height"]) |
        (refuge_sample_audit["image_width"] != refuge_sample_audit["mask_width"])
    ).sum()
)

mask_value_sets = refuge_sample_audit["mask_unique_values"].value_counts().to_dict()

print(f"REFUGE audited samples: {len(refuge_sample_audit)}")
print(f"Shape mismatches: {shape_mismatch}")
print(f"Mask value sets: {mask_value_sets}")
print(f"Sample audit: {refuge_sample_audit_csv}")

if shape_mismatch > 0:
    raise RuntimeError("Image-mask shape mismatches found in REFUGE sample audit.")

print("Status: PASS")

In [ ]:
# Cell 7: PAPILA folder and file audit

PAPILA_ROOT = Path(
    training_plan.loc[training_plan["dataset"] == "PAPILA", "root"].iloc[0]
)

papila_expected_dirs = {
    "fundus_images": PAPILA_ROOT / "FundusImages",
    "contours": PAPILA_ROOT / "ExpertsSegmentations" / "Contours",
    "images_with_contours": PAPILA_ROOT / "ExpertsSegmentations" / "ImagesWithContours",
}

papila_rows = []

for folder_key, folder_path in papila_expected_dirs.items():
    if not folder_path.exists():
        papila_rows.append(
            {
                "folder_key": folder_key,
                "path": str(folder_path),
                "exists": False,
                "n_files": 0,
                "extensions": "{}",
                "sample_files": "",
            }
        )
        continue

    files = sorted([p for p in folder_path.rglob("*") if p.is_file()])
    ext_counts = {}

    for p in files:
        ext_counts[p.suffix.lower()] = ext_counts.get(p.suffix.lower(), 0) + 1

    papila_rows.append(
        {
            "folder_key": folder_key,
            "path": str(folder_path),
            "exists": True,
            "n_files": len(files),
            "extensions": json.dumps(ext_counts, sort_keys=True),
            "sample_files": " | ".join([p.name for p in files[:12]]),
        }
    )

papila_folder_audit = pd.DataFrame(papila_rows)

papila_folder_audit_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_papila_folder_audit.csv"
papila_folder_audit.to_csv(papila_folder_audit_csv, index=False)

missing_folders = papila_folder_audit.loc[~papila_folder_audit["exists"], "folder_key"].tolist()

print(papila_folder_audit[["folder_key", "exists", "n_files", "extensions", "sample_files"]].to_string(index=False))
print(f"PAPILA audit: {papila_folder_audit_csv}")

if missing_folders:
    raise FileNotFoundError(f"Missing PAPILA folders: {missing_folders}")

if int(papila_folder_audit.loc[papila_folder_audit["folder_key"] == "fundus_images", "n_files"].iloc[0]) == 0:
    raise RuntimeError("No PAPILA fundus images found.")

if int(papila_folder_audit.loc[papila_folder_audit["folder_key"] == "contours", "n_files"].iloc[0]) == 0:
    raise RuntimeError("No PAPILA contour files found.")

print("Status: PASS")

In [ ]:
# Cell 8: Build PAPILA image-contour manifest

import re

papila_image_dir = papila_expected_dirs["fundus_images"]
papila_contour_dir = papila_expected_dirs["contours"]

papila_images = sorted(
    [p for p in papila_image_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
)

papila_contours = sorted(
    [p for p in papila_contour_dir.iterdir() if p.is_file() and p.suffix.lower() == ".txt"]
)

contour_pattern = re.compile(r"^(?P<base>.+)_(?P<target>cup|disc)_exp(?P<expert>[0-9]+)$", re.IGNORECASE)

contour_map = {}
unparsed_contours = []

for contour_path in papila_contours:
    match = contour_pattern.match(contour_path.stem)

    if match is None:
        unparsed_contours.append(contour_path.name)
        continue

    base = match.group("base")
    target = match.group("target").lower()
    expert = f"exp{match.group('expert')}"

    contour_map[(base, target, expert)] = contour_path

papila_rows = []

for image_path in papila_images:
    base = image_path.stem

    disc_exp1 = contour_map.get((base, "disc", "exp1"))
    disc_exp2 = contour_map.get((base, "disc", "exp2"))
    cup_exp1 = contour_map.get((base, "cup", "exp1"))
    cup_exp2 = contour_map.get((base, "cup", "exp2"))

    papila_rows.append(
        {
            "dataset": "PAPILA",
            "sample_id": f"PAPILA_{base}",
            "image_name": image_path.name,
            "image_path": str(image_path),
            "disc_exp1_path": str(disc_exp1) if disc_exp1 is not None else "",
            "disc_exp2_path": str(disc_exp2) if disc_exp2 is not None else "",
            "cup_exp1_path": str(cup_exp1) if cup_exp1 is not None else "",
            "cup_exp2_path": str(cup_exp2) if cup_exp2 is not None else "",
            "has_disc_exp1": disc_exp1 is not None,
            "has_disc_exp2": disc_exp2 is not None,
            "has_cup_exp1": cup_exp1 is not None,
            "has_cup_exp2": cup_exp2 is not None,
        }
    )

papila_manifest = pd.DataFrame(papila_rows)

papila_manifest["has_all_disc_contours"] = (
    papila_manifest["has_disc_exp1"] & papila_manifest["has_disc_exp2"]
)

papila_manifest["has_all_cup_contours"] = (
    papila_manifest["has_cup_exp1"] & papila_manifest["has_cup_exp2"]
)

papila_manifest["has_all_contours"] = (
    papila_manifest["has_all_disc_contours"] & papila_manifest["has_all_cup_contours"]
)

matched_contour_paths = set()

for col in ["disc_exp1_path", "disc_exp2_path", "cup_exp1_path", "cup_exp2_path"]:
    matched_contour_paths.update(
        p for p in papila_manifest[col].tolist()
        if isinstance(p, str) and len(p) > 0
    )

extra_contours = [
    p.name for p in papila_contours
    if str(p) not in matched_contour_paths and p.name not in unparsed_contours
]

papila_manifest_dir = PATHS["project_data"] / "external_datasets" / "PAPILA"
papila_manifest_dir.mkdir(parents=True, exist_ok=True)

papila_manifest_csv = papila_manifest_dir / "papila_od_oc_manifest.csv"
papila_manifest_results_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_papila_od_oc_manifest.csv"

papila_manifest.to_csv(papila_manifest_csv, index=False)
papila_manifest.to_csv(papila_manifest_results_csv, index=False)

papila_unparsed_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_papila_unparsed_txt_files.csv"
pd.DataFrame({"file_name": unparsed_contours}).to_csv(papila_unparsed_csv, index=False)

papila_manifest_audit = pd.DataFrame(
    [
        {
            "check": "fundus_images",
            "value": len(papila_manifest),
            "expected": 488,
            "passed": len(papila_manifest) == 488,
            "severity": "required",
        },
        {
            "check": "all_disc_contours",
            "value": int(papila_manifest["has_all_disc_contours"].sum()),
            "expected": 488,
            "passed": int(papila_manifest["has_all_disc_contours"].sum()) == 488,
            "severity": "required",
        },
        {
            "check": "all_cup_contours",
            "value": int(papila_manifest["has_all_cup_contours"].sum()),
            "expected": 488,
            "passed": int(papila_manifest["has_all_cup_contours"].sum()) == 488,
            "severity": "required",
        },
        {
            "check": "extra_contours",
            "value": len(extra_contours),
            "expected": 0,
            "passed": len(extra_contours) == 0,
            "severity": "required",
        },
        {
            "check": "unparsed_txt_files",
            "value": len(unparsed_contours),
            "expected": 0,
            "passed": len(unparsed_contours) == 0,
            "severity": "warning",
        },
    ]
)

papila_manifest_audit_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_papila_manifest_audit.csv"
papila_manifest_audit.to_csv(papila_manifest_audit_csv, index=False)

required_failed_checks = papila_manifest_audit.loc[
    (~papila_manifest_audit["passed"]) &
    (papila_manifest_audit["severity"] == "required"),
    "check",
].tolist()

print(f"PAPILA samples: {len(papila_manifest)}")
print(f"Complete disc contours: {int(papila_manifest['has_all_disc_contours'].sum())}")
print(f"Complete cup contours: {int(papila_manifest['has_all_cup_contours'].sum())}")
print(f"Unparsed txt files: {len(unparsed_contours)}")
print(f"PAPILA manifest: {papila_manifest_csv}")

if required_failed_checks:
    raise RuntimeError(f"PAPILA required checks failed: {required_failed_checks}")

print("Status: PASS")

In [ ]:
# Cell 9: Build RIGA original-marking manifest

RIGA_ROOT = Path(
    training_plan.loc[training_plan["dataset"] == "RIGA", "root"].iloc[0]
)

riga_source_roots = {
    "MESSIDOR": RIGA_ROOT / "MESSIDOR",
    "BinRushedcorrected": RIGA_ROOT / "BinRushedcorrected" / "BinRushed",
    "Magrabia": RIGA_ROOT / "Magrabia",
}

canonical_subgroups = {
    ("MESSIDOR", "MESSIDOR"): 460,
    ("BinRushedcorrected", "BinRushed1-Corrected"): 50,
    ("BinRushedcorrected", "BinRushed2"): 47,
    ("BinRushedcorrected", "BinRushed3"): 47,
    ("BinRushedcorrected", "BinRushed4"): 51,
    ("Magrabia", "MagrabiFemale"): 48,
    ("Magrabia", "MagrabiaMale"): 47,
}

mark_pattern = re.compile(r"^image(?P<case>[0-9]+)-(?P<mark>[1-6])$", re.IGNORECASE)
original_pattern = re.compile(r"^image(?P<case>[0-9]+)(prime)?$", re.IGNORECASE)

riga_rows = []
subgroup_audit_rows = []

for (source, subgroup), expected_cases in canonical_subgroups.items():
    source_root = riga_source_roots[source]
    subgroup_dir = source_root if subgroup == source else source_root / subgroup

    if not subgroup_dir.exists():
        raise FileNotFoundError(f"Missing RIGA subgroup directory: {subgroup_dir}")

    files = sorted(
        [p for p in subgroup_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
    )

    originals = {}
    markings = {}

    for path in files:
        stem = path.stem

        mark_match = mark_pattern.match(stem)
        if mark_match is not None:
            case_id = int(mark_match.group("case"))
            mark_id = int(mark_match.group("mark"))
            markings.setdefault(case_id, {})[mark_id] = path
            continue

        original_match = original_pattern.match(stem)
        if original_match is not None:
            case_id = int(original_match.group("case"))
            originals[case_id] = path

    case_ids = sorted(set(originals.keys()) | set(markings.keys()))

    for case_id in case_ids:
        case_markings = markings.get(case_id, {})
        row = {
            "dataset": "RIGA",
            "source": source,
            "subgroup": subgroup,
            "sample_id": f"RIGA_{source}_{subgroup}_case{case_id:04d}".replace("/", "_"),
            "case_id": case_id,
            "original_name": originals.get(case_id, Path("")).name,
            "original_path": str(originals.get(case_id, "")),
            "has_original": case_id in originals,
            "n_expert_markings": sum(mark_id in case_markings for mark_id in range(1, 7)),
            "complete_six_markings": all(mark_id in case_markings for mark_id in range(1, 7)),
        }

        for mark_id in range(1, 7):
            row[f"expert_marking_{mark_id}_path"] = str(case_markings.get(mark_id, ""))

        riga_rows.append(row)

    subgroup_audit_rows.append(
        {
            "source": source,
            "subgroup": subgroup,
            "expected_cases": expected_cases,
            "detected_cases": len(case_ids),
            "originals": len(originals),
            "cases_with_markings": len(markings),
        }
    )

riga_manifest = pd.DataFrame(riga_rows).sort_values(
    ["source", "subgroup", "case_id"]
).reset_index(drop=True)

riga_subgroup_audit = pd.DataFrame(subgroup_audit_rows)

riga_manifest_dir = PATHS["project_data"] / "external_datasets" / "RIGA"
riga_manifest_dir.mkdir(parents=True, exist_ok=True)

riga_manifest_csv = riga_manifest_dir / "riga_od_oc_manifest.csv"
riga_manifest_results_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_riga_od_oc_manifest.csv"
riga_subgroup_audit_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_riga_subgroup_audit.csv"

riga_manifest.to_csv(riga_manifest_csv, index=False)
riga_manifest.to_csv(riga_manifest_results_csv, index=False)
riga_subgroup_audit.to_csv(riga_subgroup_audit_csv, index=False)

missing_originals = int((~riga_manifest["has_original"]).sum())
incomplete_markings = int((~riga_manifest["complete_six_markings"]).sum())
subgroup_count_errors = int(
    (riga_subgroup_audit["expected_cases"] != riga_subgroup_audit["detected_cases"]).sum()
)

print(f"RIGA samples: {len(riga_manifest)}")
print(f"Missing originals: {missing_originals}")
print(f"Incomplete six-expert markings: {incomplete_markings}")
print(f"Subgroup count errors: {subgroup_count_errors}")
print(f"RIGA manifest: {riga_manifest_csv}")

if len(riga_manifest) != 750:
    raise RuntimeError(f"Expected 750 RIGA samples, found {len(riga_manifest)}")

if missing_originals > 0:
    raise RuntimeError("Missing RIGA original images found.")

if incomplete_markings > 0:
    raise RuntimeError("Incomplete RIGA expert markings found.")

if subgroup_count_errors > 0:
    raise RuntimeError("Unexpected RIGA subgroup counts found.")

print("Status: PASS")

In [ ]:
# Cell 13: Diagnose missing RIGA original-case matches

canonical_files = riga_files_parsed[
    riga_files_parsed.apply(
        lambda row: (row["source"], row["subgroup"]) in canonical_riga_subgroups,
        axis=1,
    )
].copy()

canonical_originals = riga_original_candidates[
    riga_original_candidates.apply(
        lambda row: (row["source"], row["subgroup"]) in canonical_riga_subgroups,
        axis=1,
    )
].copy()

def parse_riga_original_case(file_name):
    stem = Path(file_name).stem
    match = re.match(r"^image(?P<case>[0-9]+)prime$", stem, re.IGNORECASE)
    if match is not None:
        return int(match.group("case"))
    return np.nan

canonical_originals["parsed_case_id"] = canonical_originals["file_name"].apply(parse_riga_original_case)

mark_cases = (
    canonical_files.groupby(["source", "subgroup", "case_id"])
    .size()
    .reset_index(name="n_mark_files")
)

original_cases = (
    canonical_originals.dropna(subset=["parsed_case_id"])
    .assign(parsed_case_id=lambda df: df["parsed_case_id"].astype(int))
    [["source", "subgroup", "parsed_case_id", "file_name", "path"]]
    .rename(columns={"parsed_case_id": "case_id", "file_name": "original_name", "path": "original_path"})
)

case_match_audit = mark_cases.merge(
    original_cases,
    on=["source", "subgroup", "case_id"],
    how="left",
)

missing_original_cases = case_match_audit[case_match_audit["original_path"].isna()].copy()
unparsed_originals = canonical_originals[canonical_originals["parsed_case_id"].isna()].copy()

missing_original_cases_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_riga_missing_original_cases.csv"
unparsed_originals_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_riga_unparsed_original_files.csv"

missing_original_cases.to_csv(missing_original_cases_csv, index=False)
unparsed_originals.to_csv(unparsed_originals_csv, index=False)

print(f"Canonical mark cases: {len(mark_cases)}")
print(f"Parsed original cases: {len(original_cases)}")
print(f"Missing original matches: {len(missing_original_cases)}")
print(f"Unparsed original files: {len(unparsed_originals)}")

if len(missing_original_cases) > 0:
    print("\nMissing original cases:")
    print(missing_original_cases[["source", "subgroup", "case_id", "n_mark_files"]].to_string(index=False))

if len(unparsed_originals) > 0:
    print("\nUnparsed original files:")
    print(unparsed_originals[["source", "subgroup", "file_name"]].to_string(index=False))

print(f"Missing cases CSV: {missing_original_cases_csv}")
print(f"Unparsed originals CSV: {unparsed_originals_csv}")

print("Status: PASS")

In [ ]:
# Cell 10: Build external segmentation manifest index

manifest_specs = [
    {
        "dataset": "REFUGE",
        "manifest_path": refuge_manifest_csv,
        "status": "ready",
        "annotation_source": "mask_images",
        "n_expected_samples": 1200,
    },
    {
        "dataset": "PAPILA",
        "manifest_path": papila_manifest_csv,
        "status": "ready",
        "annotation_source": "expert_contours_txt",
        "n_expected_samples": 488,
    },
    {
        "dataset": "RIGA",
        "manifest_path": riga_manifest_csv,
        "status": "ready",
        "annotation_source": "six_expert_marked_images",
        "n_expected_samples": 750,
    },
    {
        "dataset": "G1020",
        "manifest_path": "",
        "status": "pending_labelme_json",
        "annotation_source": "labelme_json_missing_locally",
        "n_expected_samples": 1020,
    },
]

manifest_index_rows = []

for spec in manifest_specs:
    manifest_path = Path(spec["manifest_path"]) if spec["manifest_path"] else None

    if manifest_path is not None and manifest_path.exists():
        manifest_df = pd.read_csv(manifest_path)
        n_samples = len(manifest_df)
        exists = True
    else:
        n_samples = 0
        exists = False

    manifest_index_rows.append(
        {
            "dataset": spec["dataset"],
            "manifest_path": str(manifest_path) if manifest_path is not None else "",
            "manifest_exists": exists,
            "status": spec["status"],
            "annotation_source": spec["annotation_source"],
            "n_samples": int(n_samples),
            "n_expected_samples": int(spec["n_expected_samples"]),
            "count_matches_expected": n_samples == spec["n_expected_samples"],
        }
    )

external_segmentation_manifest_index = pd.DataFrame(manifest_index_rows)

manifest_index_csv = PATHS["project_data"] / "external_datasets" / "external_segmentation_manifest_index.csv"
manifest_index_results_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_external_segmentation_manifest_index.csv"

external_segmentation_manifest_index.to_csv(manifest_index_csv, index=False)
external_segmentation_manifest_index.to_csv(manifest_index_results_csv, index=False)

ready_datasets = external_segmentation_manifest_index.loc[
    external_segmentation_manifest_index["status"] == "ready",
    "dataset",
].tolist()

failed_ready_counts = external_segmentation_manifest_index.loc[
    (external_segmentation_manifest_index["status"] == "ready") &
    (~external_segmentation_manifest_index["count_matches_expected"]),
    "dataset",
].tolist()

print(f"Ready segmentation datasets: {ready_datasets}")
print(f"Pending datasets: {external_segmentation_manifest_index.loc[external_segmentation_manifest_index['status'] != 'ready', 'dataset'].tolist()}")
print(f"Manifest index: {manifest_index_csv}")

if len(ready_datasets) != 3:
    raise RuntimeError(f"Expected 3 ready segmentation datasets, found {len(ready_datasets)}")

if failed_ready_counts:
    raise RuntimeError(f"Ready dataset count mismatch: {failed_ready_counts}")

print("Status: PASS")

In [ ]:
# Cell 11: Annotation format smoke test

import cv2

smoke_rows = []

# REFUGE
refuge_sample = refuge_manifest.iloc[0]
refuge_image = cv2.imread(refuge_sample["image_path"], cv2.IMREAD_COLOR)
refuge_mask = cv2.imread(refuge_sample["mask_path"], cv2.IMREAD_UNCHANGED)

if refuge_image is None or refuge_mask is None:
    raise RuntimeError("REFUGE smoke test failed.")

if refuge_mask.ndim == 3:
    refuge_mask_gray = cv2.cvtColor(refuge_mask, cv2.COLOR_BGR2GRAY)
else:
    refuge_mask_gray = refuge_mask

smoke_rows.append(
    {
        "dataset": "REFUGE",
        "sample_id": refuge_sample["sample_id"],
        "image_shape": str(refuge_image.shape),
        "annotation_shape": str(refuge_mask_gray.shape),
        "annotation_summary": "|".join(map(str, sorted(np.unique(refuge_mask_gray).astype(int).tolist()))),
        "status": "ok",
    }
)

# PAPILA
papila_sample = papila_manifest.iloc[0]
papila_image = cv2.imread(papila_sample["image_path"], cv2.IMREAD_COLOR)

if papila_image is None:
    raise RuntimeError("PAPILA image smoke test failed.")

papila_contour_files = [
    papila_sample["disc_exp1_path"],
    papila_sample["disc_exp2_path"],
    papila_sample["cup_exp1_path"],
    papila_sample["cup_exp2_path"],
]

papila_contour_lengths = []

for contour_path in papila_contour_files:
    contour_df = pd.read_csv(contour_path, header=None)
    papila_contour_lengths.append(len(contour_df))

smoke_rows.append(
    {
        "dataset": "PAPILA",
        "sample_id": papila_sample["sample_id"],
        "image_shape": str(papila_image.shape),
        "annotation_shape": "txt_contours",
        "annotation_summary": "|".join(map(str, papila_contour_lengths)),
        "status": "ok",
    }
)

# RIGA
riga_sample = riga_manifest.iloc[0]
riga_original = cv2.imread(riga_sample["original_path"], cv2.IMREAD_COLOR)
riga_marking = cv2.imread(riga_sample["expert_marking_1_path"], cv2.IMREAD_COLOR)

if riga_original is None or riga_marking is None:
    raise RuntimeError("RIGA smoke test failed.")

smoke_rows.append(
    {
        "dataset": "RIGA",
        "sample_id": riga_sample["sample_id"],
        "image_shape": str(riga_original.shape),
        "annotation_shape": str(riga_marking.shape),
        "annotation_summary": "expert_marked_image",
        "status": "ok",
    }
)

annotation_smoke_test = pd.DataFrame(smoke_rows)

annotation_smoke_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_annotation_format_smoke_test.csv"
annotation_smoke_test.to_csv(annotation_smoke_csv, index=False)

print(annotation_smoke_test[["dataset", "image_shape", "annotation_shape", "annotation_summary", "status"]].to_string(index=False))
print(f"Smoke test: {annotation_smoke_csv}")

if not annotation_smoke_test["status"].eq("ok").all():
    raise RuntimeError("Annotation smoke test failed.")

print("Status: PASS")

In [ ]:
# Cell 12: PAPILA contour format diagnostic

def extract_xy_points_from_txt(path):
    lines = Path(path).read_text(encoding="utf-8", errors="ignore").splitlines()
    points = []

    for line in lines:
        values = np.fromstring(line, sep=" ")
        if len(values) >= 2:
            points.append((float(values[0]), float(values[1])))

    return lines, points

papila_diagnostic_rows = []

for contour_label, contour_path in {
    "disc_exp1": papila_sample["disc_exp1_path"],
    "disc_exp2": papila_sample["disc_exp2_path"],
    "cup_exp1": papila_sample["cup_exp1_path"],
    "cup_exp2": papila_sample["cup_exp2_path"],
}.items():
    raw_lines, xy_points = extract_xy_points_from_txt(contour_path)

    papila_diagnostic_rows.append(
        {
            "sample_id": papila_sample["sample_id"],
            "contour_label": contour_label,
            "path": contour_path,
            "n_raw_lines": len(raw_lines),
            "n_xy_points": len(xy_points),
            "first_raw_line": raw_lines[0] if raw_lines else "",
            "first_xy_point": str(xy_points[0]) if xy_points else "",
            "status": "ready" if len(xy_points) >= 3 else "needs_review",
        }
    )

papila_contour_diagnostic = pd.DataFrame(papila_diagnostic_rows)

papila_contour_diagnostic_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_papila_contour_format_diagnostic.csv"
papila_contour_diagnostic.to_csv(papila_contour_diagnostic_csv, index=False)

print(
    papila_contour_diagnostic[
        ["contour_label", "n_raw_lines", "n_xy_points", "first_xy_point", "status"]
    ].to_string(index=False)
)
print(f"PAPILA contour diagnostic: {papila_contour_diagnostic_csv}")

if papila_contour_diagnostic["status"].eq("needs_review").any():
    raise RuntimeError("PAPILA contour format still needs review.")

print("Status: PASS")

In [ ]:
# Cell 13: PAPILA single-sample mask conversion smoke test

def read_papila_contour(path):
    points = []

    for line in Path(path).read_text(encoding="utf-8", errors="ignore").splitlines():
        values = np.fromstring(line, sep=" ")
        if len(values) >= 2:
            points.append([values[0], values[1]])

    if len(points) < 3:
        raise ValueError(f"Invalid contour file: {path}")

    return np.array(points, dtype=np.float32)

def rasterize_polygon(shape_hw, points):
    mask = np.zeros(shape_hw, dtype=np.uint8)
    polygon = np.round(points).astype(np.int32)
    cv2.fillPoly(mask, [polygon], 1)
    return mask

papila_sample = papila_manifest.iloc[0]
papila_image = cv2.imread(papila_sample["image_path"], cv2.IMREAD_COLOR)

if papila_image is None:
    raise RuntimeError(f"Unreadable PAPILA image: {papila_sample['image_path']}")

height, width = papila_image.shape[:2]

disc_exp1 = read_papila_contour(papila_sample["disc_exp1_path"])
disc_exp2 = read_papila_contour(papila_sample["disc_exp2_path"])
cup_exp1 = read_papila_contour(papila_sample["cup_exp1_path"])
cup_exp2 = read_papila_contour(papila_sample["cup_exp2_path"])

disc_mask_exp1 = rasterize_polygon((height, width), disc_exp1)
disc_mask_exp2 = rasterize_polygon((height, width), disc_exp2)
cup_mask_exp1 = rasterize_polygon((height, width), cup_exp1)
cup_mask_exp2 = rasterize_polygon((height, width), cup_exp2)

disc_consensus = ((disc_mask_exp1 + disc_mask_exp2) >= 1).astype(np.uint8)
cup_consensus = ((cup_mask_exp1 + cup_mask_exp2) >= 1).astype(np.uint8)

cup_outside_disc = int(((cup_consensus == 1) & (disc_consensus == 0)).sum())

papila_mask_smoke = pd.DataFrame(
    [
        {
            "sample_id": papila_sample["sample_id"],
            "image_height": height,
            "image_width": width,
            "disc_exp1_points": len(disc_exp1),
            "disc_exp2_points": len(disc_exp2),
            "cup_exp1_points": len(cup_exp1),
            "cup_exp2_points": len(cup_exp2),
            "disc_area_ratio": float(disc_consensus.mean()),
            "cup_area_ratio": float(cup_consensus.mean()),
            "cup_outside_disc_pixels": cup_outside_disc,
            "status": "ok",
        }
    ]
)

papila_mask_smoke_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_papila_mask_conversion_smoke_test.csv"
papila_mask_smoke.to_csv(papila_mask_smoke_csv, index=False)

print(f"Sample: {papila_sample['sample_id']}")
print(f"Disc area ratio: {papila_mask_smoke['disc_area_ratio'].iloc[0]:.5f}")
print(f"Cup area ratio: {papila_mask_smoke['cup_area_ratio'].iloc[0]:.5f}")
print(f"Cup outside disc pixels: {cup_outside_disc}")
print(f"Smoke test: {papila_mask_smoke_csv}")

if disc_consensus.sum() == 0:
    raise RuntimeError("Empty PAPILA disc mask.")

if cup_consensus.sum() == 0:
    raise RuntimeError("Empty PAPILA cup mask.")

print("Status: PASS")

In [ ]:
# Cell 14: RIGA color-rule visual diagnostic

import matplotlib.pyplot as plt

RIGA_DIAGNOSTIC_DIR = PATHS["results_04_od_segmentor_external_validation"] / "riga_visual_diagnostics"
RIGA_DIAGNOSTIC_DIR.mkdir(parents=True, exist_ok=True)

def extract_riga_color_lines(original_bgr, marked_bgr):
    if original_bgr.shape != marked_bgr.shape:
        marked_bgr = cv2.resize(
            marked_bgr,
            (original_bgr.shape[1], original_bgr.shape[0]),
            interpolation=cv2.INTER_AREA,
        )

    diff = cv2.absdiff(original_bgr, marked_bgr)
    diff_gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(marked_bgr, cv2.COLOR_BGR2HSV)

    base = (diff_gray > 25) & (hsv[:, :, 1] > 60) & (hsv[:, :, 2] > 40)

    od_line = (
        base &
        (
            ((hsv[:, :, 0] >= 0) & (hsv[:, :, 0] <= 25)) |
            (hsv[:, :, 0] >= 170)
        )
    ).astype(np.uint8)

    oc_line = (
        base &
        (hsv[:, :, 0] >= 95) &
        (hsv[:, :, 0] <= 145)
    ).astype(np.uint8)

    return od_line, oc_line

def overlay_mask(image_bgr, mask, color_bgr):
    out = image_bgr.copy()
    out[mask > 0] = (0.6 * out[mask > 0] + 0.4 * np.array(color_bgr)).astype(np.uint8)
    return out

def resize_panel(image_bgr, width=512):
    h, w = image_bgr.shape[:2]
    return cv2.resize(image_bgr, (width, int(h * width / w)), interpolation=cv2.INTER_AREA)

def add_label(image_bgr, label):
    out = image_bgr.copy()
    cv2.putText(out, label, (12, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(out, label, (12, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 1, cv2.LINE_AA)
    return out

riga_sample = riga_manifest.iloc[0]
riga_original = cv2.imread(riga_sample["original_path"], cv2.IMREAD_COLOR)
riga_marked = cv2.imread(riga_sample["expert_marking_1_path"], cv2.IMREAD_COLOR)

if riga_original is None:
    raise RuntimeError(f"Unreadable RIGA original: {riga_sample['original_path']}")

if riga_marked is None:
    raise RuntimeError(f"Unreadable RIGA marking: {riga_sample['expert_marking_1_path']}")

od_line, oc_line = extract_riga_color_lines(riga_original, riga_marked)

panels = [
    add_label(resize_panel(riga_original), "Original"),
    add_label(resize_panel(riga_marked), "Expert marking"),
    add_label(resize_panel(overlay_mask(riga_original, od_line, (0, 0, 255))), "OD line rule"),
    add_label(resize_panel(overlay_mask(riga_original, oc_line, (255, 0, 0))), "OC line rule"),
]

diagnostic_panel = cv2.vconcat([
    cv2.hconcat(panels[:2]),
    cv2.hconcat(panels[2:]),
])

riga_color_panel_path = RIGA_DIAGNOSTIC_DIR / "riga_od_oc_color_rule_diagnostic.png"
cv2.imwrite(str(riga_color_panel_path), diagnostic_panel)

riga_color_rule_summary = pd.DataFrame(
    [
        {
            "sample_id": riga_sample["sample_id"],
            "expert_id": 1,
            "od_line_pixels": int(od_line.sum()),
            "oc_line_pixels": int(oc_line.sum()),
            "od_color_rule": "red_orange_hue",
            "oc_color_rule": "blue_purple_hue",
            "panel_path": str(riga_color_panel_path),
        }
    ]
)

riga_color_rule_summary_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_riga_color_rule_summary.csv"
riga_color_rule_summary.to_csv(riga_color_rule_summary_csv, index=False)

panel_rgb = cv2.cvtColor(diagnostic_panel, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(12, 8))
plt.imshow(panel_rgb)
plt.axis("off")
plt.show()

print(f"OD line pixels: {int(od_line.sum())}")
print(f"OC line pixels: {int(oc_line.sum())}")
print(f"Diagnostic panel: {riga_color_panel_path}")

if od_line.sum() == 0:
    raise RuntimeError("Empty RIGA OD line mask.")

if oc_line.sum() == 0:
    raise RuntimeError("Empty RIGA OC line mask.")

print("Status: PASS")

In [ ]:
# Cell 15: RIGA single-sample OD/OC mask conversion smoke test

def fill_convex_hull_from_line(line_mask):
    line_mask = (line_mask > 0).astype(np.uint8)
    ys, xs = np.where(line_mask > 0)

    if len(xs) < 3:
        return np.zeros_like(line_mask)

    points = np.column_stack([xs, ys]).astype(np.int32)
    hull = cv2.convexHull(points)

    filled = np.zeros_like(line_mask)
    cv2.fillConvexPoly(filled, hull, 1)

    return filled

od_mask = fill_convex_hull_from_line(od_line)
oc_mask = fill_convex_hull_from_line(oc_line)

oc_outside_od = int(((oc_mask == 1) & (od_mask == 0)).sum())
oc_outside_od_ratio = float(oc_outside_od / max(int(oc_mask.sum()), 1))

riga_mask_smoke = pd.DataFrame(
    [
        {
            "sample_id": riga_sample["sample_id"],
            "expert_id": 1,
            "od_line_pixels": int(od_line.sum()),
            "oc_line_pixels": int(oc_line.sum()),
            "od_area_ratio": float(od_mask.mean()),
            "oc_area_ratio": float(oc_mask.mean()),
            "oc_outside_od_pixels": oc_outside_od,
            "oc_outside_od_ratio": oc_outside_od_ratio,
            "status": "ok",
        }
    ]
)

riga_mask_smoke_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_riga_mask_conversion_smoke_test.csv"
riga_mask_smoke.to_csv(riga_mask_smoke_csv, index=False)

print(f"Sample: {riga_sample['sample_id']}")
print(f"OD area ratio: {riga_mask_smoke['od_area_ratio'].iloc[0]:.6f}")
print(f"OC area ratio: {riga_mask_smoke['oc_area_ratio'].iloc[0]:.6f}")
print(f"OC outside OD ratio: {oc_outside_od_ratio:.6f}")
print(f"Smoke test: {riga_mask_smoke_csv}")

if od_mask.sum() == 0:
    raise RuntimeError("Empty RIGA OD mask.")

if oc_mask.sum() == 0:
    raise RuntimeError("Empty RIGA OC mask.")

if oc_mask.sum() >= od_mask.sum():
    raise RuntimeError("RIGA OC mask area is not smaller than OD mask area.")

if oc_outside_od_ratio > 0.01:
    raise RuntimeError("RIGA OC mask has excessive area outside OD mask.")

print("Status: PASS")

In [ ]:
# Cell 16: Save annotation conversion specification

conversion_specs = [
    {
        "dataset": "REFUGE",
        "input_annotation": "mask_image",
        "od_rule": "mask_value_128_or_255_pending_class_mapping",
        "oc_rule": "mask_value_128_or_255_pending_class_mapping",
        "conversion_ready": True,
        "notes": "Mask values detected: 0, 128, 255. Final OD/OC class mapping will be verified before batch conversion.",
    },
    {
        "dataset": "PAPILA",
        "input_annotation": "txt_contours",
        "od_rule": "fill polygon from disc_exp1 and disc_exp2 contours",
        "oc_rule": "fill polygon from cup_exp1 and cup_exp2 contours",
        "conversion_ready": True,
        "notes": "Scientific notation coordinates parsed with numpy.fromstring.",
    },
    {
        "dataset": "RIGA",
        "input_annotation": "expert_marked_images",
        "od_rule": "red_orange hue line extraction followed by convex hull fill",
        "oc_rule": "blue_purple hue line extraction followed by convex hull fill",
        "conversion_ready": True,
        "notes": "RIGA is used for anatomical segmentation only, not diagnostic glaucoma validation.",
    },
    {
        "dataset": "G1020",
        "input_annotation": "labelme_json",
        "od_rule": "pending local JSON annotations",
        "oc_rule": "pending local JSON annotations",
        "conversion_ready": False,
        "notes": "Local copy has images but no LabelMe JSON annotations.",
    },
]

annotation_conversion_specs = pd.DataFrame(conversion_specs)

conversion_specs_csv = PATHS["project_data"] / "external_datasets" / "annotation_conversion_specs.csv"
conversion_specs_results_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_annotation_conversion_specs.csv"

annotation_conversion_specs.to_csv(conversion_specs_csv, index=False)
annotation_conversion_specs.to_csv(conversion_specs_results_csv, index=False)

ready_count = int(annotation_conversion_specs["conversion_ready"].sum())

print(f"Conversion-ready datasets: {ready_count}/4")
print(
    annotation_conversion_specs[
        ["dataset", "input_annotation", "conversion_ready"]
    ].to_string(index=False)
)
print(f"Conversion specs: {conversion_specs_csv}")

if ready_count != 3:
    raise RuntimeError(f"Expected 3 conversion-ready datasets, found {ready_count}")

print("Status: PASS")

In [ ]:
# Cell 17: REFUGE OD/OC mask-value mapping diagnostic

rng = np.random.default_rng(int(PROJECT_CONFIG["seed"]))
refuge_mapping_rows = []

for split in ["train", "val", "test"]:
    split_df = refuge_manifest[refuge_manifest["split_original"] == split].copy()
    sample_size = min(12, len(split_df))
    sample_indices = rng.choice(split_df.index.to_numpy(), size=sample_size, replace=False)

    for idx in sample_indices:
        row = refuge_manifest.loc[idx]
        mask = cv2.imread(row["mask_path"], cv2.IMREAD_UNCHANGED)

        if mask is None:
            raise RuntimeError(f"Unreadable REFUGE mask: {row['mask_path']}")

        if mask.ndim == 3:
            mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)

        values, counts = np.unique(mask, return_counts=True)
        value_counts = {int(v): int(c) for v, c in zip(values, counts)}

        required_values = {0, 128, 255}
        missing_values = sorted(required_values - set(value_counts.keys()))

        background_pixels = value_counts.get(255, 0)
        od_pixels = value_counts.get(128, 0)
        oc_pixels = value_counts.get(0, 0)

        refuge_mapping_rows.append(
            {
                "sample_id": row["sample_id"],
                "split_original": split,
                "background_value": 255,
                "od_value": 128,
                "oc_value": 0,
                "background_pixels": int(background_pixels),
                "od_pixels": int(od_pixels),
                "oc_pixels": int(oc_pixels),
                "od_gt_oc": od_pixels > oc_pixels,
                "background_is_largest": background_pixels > max(od_pixels, oc_pixels),
                "missing_required_values": "|".join(map(str, missing_values)),
                "all_values": "|".join(map(str, sorted(value_counts.keys()))),
            }
        )

refuge_mapping_diagnostic = pd.DataFrame(refuge_mapping_rows)

refuge_mapping_summary = pd.DataFrame(
    [
        {
            "n_samples": len(refuge_mapping_diagnostic),
            "background_value": 255,
            "od_value": 128,
            "oc_value": 0,
            "background_largest_samples": int(refuge_mapping_diagnostic["background_is_largest"].sum()),
            "od_larger_than_oc_samples": int(refuge_mapping_diagnostic["od_gt_oc"].sum()),
            "samples_with_missing_values": int(refuge_mapping_diagnostic["missing_required_values"].ne("").sum()),
            "mean_background_pixels": float(refuge_mapping_diagnostic["background_pixels"].mean()),
            "mean_od_pixels": float(refuge_mapping_diagnostic["od_pixels"].mean()),
            "mean_oc_pixels": float(refuge_mapping_diagnostic["oc_pixels"].mean()),
        }
    ]
)

refuge_mapping_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_refuge_mask_value_mapping_diagnostic.csv"
refuge_mapping_summary_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_refuge_mask_value_mapping_summary.csv"

refuge_mapping_diagnostic.to_csv(refuge_mapping_csv, index=False)
refuge_mapping_summary.to_csv(refuge_mapping_summary_csv, index=False)

background_ok = bool(refuge_mapping_diagnostic["background_is_largest"].all())
missing_value_cases = int(refuge_mapping_diagnostic["missing_required_values"].ne("").sum())
od_gt_oc_count = int(refuge_mapping_diagnostic["od_gt_oc"].sum())

print(f"REFUGE mapping samples: {len(refuge_mapping_diagnostic)}")
print("Fixed mapping: background=255, OD=128, OC=0")
print(f"Background largest: {int(refuge_mapping_diagnostic['background_is_largest'].sum())}/{len(refuge_mapping_diagnostic)}")
print(f"OD area > OC area: {od_gt_oc_count}/{len(refuge_mapping_diagnostic)}")
print(f"Missing required value cases: {missing_value_cases}")
print(f"Mapping summary: {refuge_mapping_summary_csv}")

if not background_ok:
    raise RuntimeError("REFUGE background value 255 is not consistently dominant.")

if missing_value_cases > 0:
    raise RuntimeError("Some sampled REFUGE masks do not contain all required values: 0, 128, 255.")

print("Status: PASS")

In [ ]:
# Cell 18: Save final annotation conversion configuration

conversion_config = {
    "target_mask_format": {
        "background": 0,
        "optic_disc": 1,
        "optic_cup": 2,
    },
    "datasets": {
        "REFUGE": {
            "input_annotation": "mask_image",
            "background_value": 255,
            "od_value": 128,
            "oc_value": 0,
            "conversion_ready": True,
        },
        "PAPILA": {
            "input_annotation": "txt_contours",
            "od_rule": "disc_exp1_or_disc_exp2_polygon_union",
            "oc_rule": "cup_exp1_or_cup_exp2_polygon_union",
            "conversion_ready": True,
        },
        "RIGA": {
            "input_annotation": "expert_marked_images",
            "od_rule": "red_orange_hue_line_convex_hull",
            "oc_rule": "blue_purple_hue_line_convex_hull",
            "conversion_ready": True,
        },
        "G1020": {
            "input_annotation": "labelme_json",
            "conversion_ready": False,
            "reason": "Local copy does not include LabelMe JSON annotations.",
        },
    },
}

conversion_config_json = PATHS["project_data"] / "external_datasets" / "annotation_conversion_config.json"
conversion_config_results_json = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_annotation_conversion_config.json"

with open(conversion_config_json, "w", encoding="utf-8") as f:
    json.dump(conversion_config, f, indent=2)

with open(conversion_config_results_json, "w", encoding="utf-8") as f:
    json.dump(conversion_config, f, indent=2)

conversion_summary = pd.DataFrame(
    [
        {
            "dataset": dataset,
            "input_annotation": spec["input_annotation"],
            "conversion_ready": spec["conversion_ready"],
        }
        for dataset, spec in conversion_config["datasets"].items()
    ]
)

conversion_summary_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_annotation_conversion_summary.csv"
conversion_summary.to_csv(conversion_summary_csv, index=False)

ready_datasets = conversion_summary.loc[
    conversion_summary["conversion_ready"], "dataset"
].tolist()

print(f"Conversion-ready datasets: {ready_datasets}")
print(f"Conversion config: {conversion_config_json}")

if set(ready_datasets) != {"REFUGE", "PAPILA", "RIGA"}:
    raise RuntimeError(f"Unexpected conversion-ready datasets: {ready_datasets}")

print("Status: PASS")

In [ ]:
# Temporary recovery cell: reload REFUGE/PAPILA manifests and required functions

from pathlib import Path
import json

import cv2
import numpy as np
import pandas as pd

JOURNAL_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))
CONFIG_PATH = JOURNAL_ROOT / "configs" / "project_config.json"
PATH_REGISTRY_PATH = JOURNAL_ROOT / "configs" / "path_registry.csv"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)

path_registry = pd.read_csv(PATH_REGISTRY_PATH)
PATHS = {
    row["path_key"]: Path(row["path"])
    for _, row in path_registry.iterrows()
}

STANDARD_SIZE = 512

refuge_manifest_csv = (
    PATHS["project_data"] / "external_datasets" / "REFUGE" / "refuge_od_oc_manifest.csv"
)
papila_manifest_csv = (
    PATHS["project_data"] / "external_datasets" / "PAPILA" / "papila_od_oc_manifest.csv"
)

if not refuge_manifest_csv.exists():
    raise FileNotFoundError(f"Missing REFUGE manifest: {refuge_manifest_csv}")

if not papila_manifest_csv.exists():
    raise FileNotFoundError(f"Missing PAPILA manifest: {papila_manifest_csv}")

refuge_manifest = pd.read_csv(refuge_manifest_csv)
papila_manifest = pd.read_csv(papila_manifest_csv)

def read_papila_contour(path):
    points = []

    for line in Path(path).read_text(encoding="utf-8", errors="ignore").splitlines():
        values = np.fromstring(line, sep=" ")
        if len(values) >= 2:
            points.append([values[0], values[1]])

    if len(points) < 3:
        raise ValueError(f"Invalid contour file: {path}")

    return np.array(points, dtype=np.float32)

def rasterize_polygon(shape_hw, points):
    mask = np.zeros(shape_hw, dtype=np.uint8)
    polygon = np.round(points).astype(np.int32)
    cv2.fillPoly(mask, [polygon], 1)
    return mask

print(f"REFUGE manifest rows: {len(refuge_manifest)}")
print(f"PAPILA manifest rows: {len(papila_manifest)}")

if len(refuge_manifest) != 1200:
    raise RuntimeError(f"Expected 1200 REFUGE samples, found {len(refuge_manifest)}")

if len(papila_manifest) != 488:
    raise RuntimeError(f"Expected 488 PAPILA samples, found {len(papila_manifest)}")

print("Status: PASS")

In [ ]:
# Cell 19: Convert REFUGE annotations to standardized 512 masks

STANDARD_SIZE = 512

REFUGE_LOCAL_STD_DIR = Path("/content/refuge_standardized_512")
REFUGE_LOCAL_IMG_DIR = REFUGE_LOCAL_STD_DIR / "images_512"
REFUGE_LOCAL_MASK_DIR = REFUGE_LOCAL_STD_DIR / "masks_512"

REFUGE_LOCAL_IMG_DIR.mkdir(parents=True, exist_ok=True)
REFUGE_LOCAL_MASK_DIR.mkdir(parents=True, exist_ok=True)

REFUGE_DRIVE_STD_DIR = PATHS["external_datasets"] / "standardized_segmentation" / "REFUGE"
REFUGE_DRIVE_STD_DIR.mkdir(parents=True, exist_ok=True)

refuge_std_rows = []

for idx, row in refuge_manifest.reset_index(drop=True).iterrows():
    image_512_path = REFUGE_LOCAL_IMG_DIR / f"{row['sample_id']}.jpg"
    mask_512_path = REFUGE_LOCAL_MASK_DIR / f"{row['sample_id']}.png"

    if image_512_path.exists() and mask_512_path.exists():
        mask_512 = cv2.imread(str(mask_512_path), cv2.IMREAD_GRAYSCALE)
    else:
        image = cv2.imread(row["image_path"], cv2.IMREAD_COLOR)
        mask = cv2.imread(row["mask_path"], cv2.IMREAD_GRAYSCALE)

        if image is None:
            raise RuntimeError(f"Unreadable REFUGE image: {row['image_path']}")

        if mask is None:
            raise RuntimeError(f"Unreadable REFUGE mask: {row['mask_path']}")

        target_mask = np.zeros(mask.shape, dtype=np.uint8)
        target_mask[mask == 128] = 1
        target_mask[mask == 0] = 2

        image_512 = cv2.resize(image, (STANDARD_SIZE, STANDARD_SIZE), interpolation=cv2.INTER_AREA)
        mask_512 = cv2.resize(target_mask, (STANDARD_SIZE, STANDARD_SIZE), interpolation=cv2.INTER_NEAREST)

        cv2.imwrite(str(image_512_path), image_512)
        cv2.imwrite(str(mask_512_path), mask_512)

    refuge_std_rows.append(
        {
            "dataset": "REFUGE",
            "sample_id": row["sample_id"],
            "split_original": row["split_original"],
            "image_512_path": str(image_512_path),
            "mask_512_path": str(mask_512_path),
            "mask_values": "|".join(map(str, sorted(np.unique(mask_512).astype(int).tolist()))),
            "od_pixels_512": int((mask_512 == 1).sum()),
            "oc_pixels_512": int((mask_512 == 2).sum()),
        }
    )

    if (idx + 1) % 100 == 0:
        print(f"Processed REFUGE: {idx + 1}/{len(refuge_manifest)}")

refuge_standardized_manifest = pd.DataFrame(refuge_std_rows)

refuge_standardized_manifest_csv = REFUGE_DRIVE_STD_DIR / "refuge_standardized_segmentation_manifest.csv"
refuge_standardized_manifest_results_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_refuge_standardized_segmentation_manifest.csv"

refuge_standardized_manifest.to_csv(refuge_standardized_manifest_csv, index=False)
refuge_standardized_manifest.to_csv(refuge_standardized_manifest_results_csv, index=False)

n_images_512 = len(list(REFUGE_LOCAL_IMG_DIR.glob("*.jpg")))
n_masks_512 = len(list(REFUGE_LOCAL_MASK_DIR.glob("*.png")))
empty_od = int((refuge_standardized_manifest["od_pixels_512"] == 0).sum())
empty_oc = int((refuge_standardized_manifest["oc_pixels_512"] == 0).sum())

print(f"REFUGE standardized samples: {len(refuge_standardized_manifest)}")
print(f"Local images 512: {n_images_512}")
print(f"Local masks 512: {n_masks_512}")
print(f"Empty OD masks: {empty_od}")
print(f"Empty OC masks: {empty_oc}")
print(f"Manifest: {refuge_standardized_manifest_csv}")

if len(refuge_standardized_manifest) != 1200:
    raise RuntimeError("Unexpected REFUGE standardized manifest size.")

if n_images_512 != 1200 or n_masks_512 != 1200:
    raise RuntimeError("Unexpected REFUGE standardized local file count.")

if empty_od > 0:
    raise RuntimeError("Empty REFUGE OD masks found.")

print("Status: PASS")

In [ ]:
# Cell 20: Convert PAPILA annotations to standardized 512 masks

PAPILA_LOCAL_STD_DIR = Path("/content/papila_standardized_512")
PAPILA_LOCAL_IMG_DIR = PAPILA_LOCAL_STD_DIR / "images_512"
PAPILA_LOCAL_MASK_DIR = PAPILA_LOCAL_STD_DIR / "masks_512"

PAPILA_LOCAL_IMG_DIR.mkdir(parents=True, exist_ok=True)
PAPILA_LOCAL_MASK_DIR.mkdir(parents=True, exist_ok=True)

PAPILA_DRIVE_STD_DIR = PATHS["external_datasets"] / "standardized_segmentation" / "PAPILA"
PAPILA_DRIVE_STD_DIR.mkdir(parents=True, exist_ok=True)

papila_std_rows = []

for idx, row in papila_manifest.reset_index(drop=True).iterrows():
    image_512_path = PAPILA_LOCAL_IMG_DIR / f"{row['sample_id']}.jpg"
    mask_512_path = PAPILA_LOCAL_MASK_DIR / f"{row['sample_id']}.png"

    if image_512_path.exists() and mask_512_path.exists():
        mask_512 = cv2.imread(str(mask_512_path), cv2.IMREAD_GRAYSCALE)
    else:
        image = cv2.imread(row["image_path"], cv2.IMREAD_COLOR)

        if image is None:
            raise RuntimeError(f"Unreadable PAPILA image: {row['image_path']}")

        height, width = image.shape[:2]

        disc_exp1 = rasterize_polygon((height, width), read_papila_contour(row["disc_exp1_path"]))
        disc_exp2 = rasterize_polygon((height, width), read_papila_contour(row["disc_exp2_path"]))
        cup_exp1 = rasterize_polygon((height, width), read_papila_contour(row["cup_exp1_path"]))
        cup_exp2 = rasterize_polygon((height, width), read_papila_contour(row["cup_exp2_path"]))

        disc_mask = ((disc_exp1 + disc_exp2) >= 1).astype(np.uint8)
        cup_mask = ((cup_exp1 + cup_exp2) >= 1).astype(np.uint8)

        target_mask = np.zeros((height, width), dtype=np.uint8)
        target_mask[disc_mask == 1] = 1
        target_mask[cup_mask == 1] = 2

        image_512 = cv2.resize(image, (STANDARD_SIZE, STANDARD_SIZE), interpolation=cv2.INTER_AREA)
        mask_512 = cv2.resize(target_mask, (STANDARD_SIZE, STANDARD_SIZE), interpolation=cv2.INTER_NEAREST)

        cv2.imwrite(str(image_512_path), image_512)
        cv2.imwrite(str(mask_512_path), mask_512)

    papila_std_rows.append(
        {
            "dataset": "PAPILA",
            "sample_id": row["sample_id"],
            "image_512_path": str(image_512_path),
            "mask_512_path": str(mask_512_path),
            "mask_values": "|".join(map(str, sorted(np.unique(mask_512).astype(int).tolist()))),
            "od_pixels_512": int((mask_512 == 1).sum()),
            "oc_pixels_512": int((mask_512 == 2).sum()),
        }
    )

    if (idx + 1) % 100 == 0:
        print(f"Processed PAPILA: {idx + 1}/{len(papila_manifest)}")

papila_standardized_manifest = pd.DataFrame(papila_std_rows)

papila_standardized_manifest_csv = PAPILA_DRIVE_STD_DIR / "papila_standardized_segmentation_manifest.csv"
papila_standardized_manifest_results_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_papila_standardized_segmentation_manifest.csv"

papila_standardized_manifest.to_csv(papila_standardized_manifest_csv, index=False)
papila_standardized_manifest.to_csv(papila_standardized_manifest_results_csv, index=False)

n_images_512 = len(list(PAPILA_LOCAL_IMG_DIR.glob("*.jpg")))
n_masks_512 = len(list(PAPILA_LOCAL_MASK_DIR.glob("*.png")))
empty_od = int((papila_standardized_manifest["od_pixels_512"] == 0).sum())
empty_oc = int((papila_standardized_manifest["oc_pixels_512"] == 0).sum())

print(f"PAPILA standardized samples: {len(papila_standardized_manifest)}")
print(f"Local images 512: {n_images_512}")
print(f"Local masks 512: {n_masks_512}")
print(f"Empty OD masks: {empty_od}")
print(f"Empty OC masks: {empty_oc}")
print(f"Manifest: {papila_standardized_manifest_csv}")

if len(papila_standardized_manifest) != 488:
    raise RuntimeError("Unexpected PAPILA standardized manifest size.")

if n_images_512 != 488 or n_masks_512 != 488:
    raise RuntimeError("Unexpected PAPILA standardized local file count.")

if empty_od > 0:
    raise RuntimeError("Empty PAPILA OD masks found.")

print("Status: PASS")

In [ ]:
# Cell 21: Build RIGA original/expert-marking manifest

import re

RIGA_ROOT = Path(
    training_plan.loc[training_plan["dataset"] == "RIGA", "root"].iloc[0]
)

riga_source_roots = {
    "MESSIDOR": RIGA_ROOT / "MESSIDOR",
    "BinRushedcorrected": RIGA_ROOT / "BinRushedcorrected" / "BinRushed",
    "Magrabia": RIGA_ROOT / "Magrabia",
}

canonical_subgroups = {
    ("MESSIDOR", "MESSIDOR"): 460,
    ("BinRushedcorrected", "BinRushed1-Corrected"): 50,
    ("BinRushedcorrected", "BinRushed2"): 47,
    ("BinRushedcorrected", "BinRushed3"): 47,
    ("BinRushedcorrected", "BinRushed4"): 51,
    ("Magrabia", "MagrabiFemale"): 48,
    ("Magrabia", "MagrabiaMale"): 47,
}

mark_pattern = re.compile(r"^image(?P<case>[0-9]+)-(?P<expert>[1-6])$", re.IGNORECASE)
original_pattern = re.compile(r"^image(?P<case>[0-9]+)(prime)?$", re.IGNORECASE)

riga_rows = []
audit_rows = []

for (source, subgroup), expected_cases in canonical_subgroups.items():
    source_root = riga_source_roots[source]
    subgroup_dir = source_root if subgroup == source else source_root / subgroup

    if not subgroup_dir.exists():
        raise FileNotFoundError(f"Missing RIGA subgroup directory: {subgroup_dir}")

    files = sorted(
        p for p in subgroup_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

    originals = {}
    markings = {}

    for path in files:
        original_match = original_pattern.match(path.stem)
        mark_match = mark_pattern.match(path.stem)

        if mark_match is not None:
            case_id = int(mark_match.group("case"))
            expert_id = int(mark_match.group("expert"))
            markings.setdefault(case_id, {})[expert_id] = path
        elif original_match is not None:
            case_id = int(original_match.group("case"))
            originals[case_id] = path

    case_ids = sorted(set(originals) | set(markings))

    for case_id in case_ids:
        expert_paths = markings.get(case_id, {})

        row = {
            "dataset": "RIGA",
            "source": source,
            "subgroup": subgroup,
            "sample_id": f"RIGA_{source}_{subgroup}_case{case_id:04d}".replace("/", "_"),
            "case_id": case_id,
            "original_path": str(originals.get(case_id, "")),
            "has_original": case_id in originals,
            "n_expert_markings": sum(expert_id in expert_paths for expert_id in range(1, 7)),
            "complete_six_markings": all(expert_id in expert_paths for expert_id in range(1, 7)),
        }

        for expert_id in range(1, 7):
            row[f"expert_marking_{expert_id}_path"] = str(expert_paths.get(expert_id, ""))

        riga_rows.append(row)

    audit_rows.append(
        {
            "source": source,
            "subgroup": subgroup,
            "expected_cases": expected_cases,
            "detected_cases": len(case_ids),
            "originals": len(originals),
            "cases_with_markings": len(markings),
        }
    )

riga_manifest = pd.DataFrame(riga_rows).sort_values(
    ["source", "subgroup", "case_id"]
).reset_index(drop=True)

riga_manifest_audit = pd.DataFrame(audit_rows)

riga_manifest_dir = PATHS["project_data"] / "external_datasets" / "RIGA"
riga_manifest_dir.mkdir(parents=True, exist_ok=True)

riga_manifest_csv = riga_manifest_dir / "riga_od_oc_manifest.csv"
riga_manifest_results_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_riga_od_oc_manifest.csv"
riga_manifest_audit_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_riga_manifest_audit.csv"

riga_manifest.to_csv(riga_manifest_csv, index=False)
riga_manifest.to_csv(riga_manifest_results_csv, index=False)
riga_manifest_audit.to_csv(riga_manifest_audit_csv, index=False)

missing_originals = int((~riga_manifest["has_original"]).sum())
incomplete_markings = int((~riga_manifest["complete_six_markings"]).sum())
count_errors = int((riga_manifest_audit["expected_cases"] != riga_manifest_audit["detected_cases"]).sum())

print(f"RIGA samples: {len(riga_manifest)}")
print(f"Missing originals: {missing_originals}")
print(f"Incomplete expert markings: {incomplete_markings}")
print(f"Subgroup count errors: {count_errors}")
print(f"RIGA manifest: {riga_manifest_csv}")

if len(riga_manifest) != 750:
    raise RuntimeError(f"Expected 750 RIGA samples, found {len(riga_manifest)}")

if missing_originals > 0:
    raise RuntimeError("Missing RIGA original images found.")

if incomplete_markings > 0:
    raise RuntimeError("Incomplete RIGA expert markings found.")

if count_errors > 0:
    raise RuntimeError("Unexpected RIGA subgroup counts found.")

print("Status: PASS")

In [ ]:
# Cell 22: RIGA difference-based mask reconstruction smoke test

import cv2

STANDARD_SIZE = 512

def resize_to_512(image):
    return cv2.resize(image, (STANDARD_SIZE, STANDARD_SIZE), interpolation=cv2.INTER_AREA)

def fill_hull_from_binary(mask):
    ys, xs = np.where(mask > 0)

    if len(xs) < 3:
        return np.zeros_like(mask, dtype=np.uint8)

    points = np.column_stack([xs, ys]).astype(np.int32)
    hull = cv2.convexHull(points)

    filled = np.zeros_like(mask, dtype=np.uint8)
    cv2.fillConvexPoly(filled, hull, 1)

    return filled

def reconstruct_riga_expert_masks(original_bgr, marked_bgr, diff_threshold=5):
    original_512 = resize_to_512(original_bgr)
    marked_512 = resize_to_512(marked_bgr)

    diff = cv2.absdiff(original_512, marked_512)
    diff_gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)

    line_mask = (diff_gray > diff_threshold).astype(np.uint8)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(line_mask, 8)

    components = []

    for label_id in range(1, num_labels):
        line_area = int(stats[label_id, cv2.CC_STAT_AREA])

        if line_area < 5:
            continue

        component = (labels == label_id).astype(np.uint8)
        filled = fill_hull_from_binary(component)
        filled_area = int(filled.sum())

        if filled_area < 20:
            continue

        components.append(
            {
                "line_area": line_area,
                "filled_area": filled_area,
                "mask": filled,
            }
        )

    components = sorted(components, key=lambda x: x["line_area"], reverse=True)

    if len(components) < 2:
        return None, None, len(components)

    od_mask = components[0]["mask"]
    oc_mask = components[1]["mask"]
    oc_mask = (oc_mask & od_mask).astype(np.uint8)

    return od_mask, oc_mask, len(components)

def consensus_mask(mask_list, min_votes=3):
    vote = np.stack(mask_list, axis=0).sum(axis=0)

    for threshold in [min_votes, 2, 1]:
        consensus = (vote >= threshold).astype(np.uint8)
        if consensus.sum() > 0:
            return consensus, threshold

    return np.zeros_like(mask_list[0], dtype=np.uint8), 0

sample_riga = (
    riga_manifest
    .groupby(["source", "subgroup"], group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

smoke_rows = []

for _, row in sample_riga.iterrows():
    original = cv2.imread(row["original_path"], cv2.IMREAD_COLOR)

    if original is None:
        raise RuntimeError(f"Unreadable RIGA original: {row['original_path']}")

    od_masks = []
    oc_masks = []
    candidate_counts = []

    for expert_id in range(1, 7):
        marked = cv2.imread(row[f"expert_marking_{expert_id}_path"], cv2.IMREAD_COLOR)

        if marked is None:
            raise RuntimeError(f"Unreadable RIGA marking: {row[f'expert_marking_{expert_id}_path']}")

        od_mask, oc_mask, n_candidates = reconstruct_riga_expert_masks(original, marked)
        candidate_counts.append(n_candidates)

        if od_mask is not None and oc_mask is not None and od_mask.sum() > 0 and oc_mask.sum() > 0:
            od_masks.append(od_mask)
            oc_masks.append(oc_mask)

    if len(od_masks) == 0 or len(oc_masks) == 0:
        od_consensus = np.zeros((STANDARD_SIZE, STANDARD_SIZE), dtype=np.uint8)
        oc_consensus = np.zeros((STANDARD_SIZE, STANDARD_SIZE), dtype=np.uint8)
        od_threshold = 0
        oc_threshold = 0
    else:
        od_consensus, od_threshold = consensus_mask(od_masks, min_votes=3)
        oc_consensus, oc_threshold = consensus_mask(oc_masks, min_votes=3)
        oc_consensus = (oc_consensus & od_consensus).astype(np.uint8)

    smoke_rows.append(
        {
            "sample_id": row["sample_id"],
            "source": row["source"],
            "subgroup": row["subgroup"],
            "valid_experts": len(od_masks),
            "min_candidates": int(min(candidate_counts)),
            "max_candidates": int(max(candidate_counts)),
            "od_pixels_512": int(od_consensus.sum()),
            "oc_pixels_512": int(oc_consensus.sum()),
            "od_threshold": od_threshold,
            "oc_threshold": oc_threshold,
        }
    )

riga_reconstruction_smoke = pd.DataFrame(smoke_rows)

riga_reconstruction_smoke_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_riga_reconstruction_smoke_test.csv"
riga_reconstruction_smoke.to_csv(riga_reconstruction_smoke_csv, index=False)

empty_od = int((riga_reconstruction_smoke["od_pixels_512"] == 0).sum())
empty_oc = int((riga_reconstruction_smoke["oc_pixels_512"] == 0).sum())

print(f"Smoke samples: {len(riga_reconstruction_smoke)}")
print(f"Empty OD masks: {empty_od}")
print(f"Empty OC masks: {empty_oc}")
print(
    riga_reconstruction_smoke
    .groupby(["source", "subgroup"])
    .agg(
        samples=("sample_id", "count"),
        empty_od=("od_pixels_512", lambda x: int((x == 0).sum())),
        empty_oc=("oc_pixels_512", lambda x: int((x == 0).sum())),
        mean_valid_experts=("valid_experts", "mean"),
    )
    .reset_index()
    .to_string(index=False)
)
print(f"Smoke test: {riga_reconstruction_smoke_csv}")

if empty_od > 0:
    raise RuntimeError("Empty OD masks found in RIGA smoke test.")

if empty_oc > 0:
    raise RuntimeError("Empty OC masks found in RIGA smoke test.")

print("Status: PASS")

In [ ]:
# Cell 23: Convert RIGA annotations to standardized 512 masks

RIGA_LOCAL_STD_DIR = Path("/content/riga_standardized_512")
RIGA_LOCAL_IMG_DIR = RIGA_LOCAL_STD_DIR / "images_512"
RIGA_LOCAL_MASK_DIR = RIGA_LOCAL_STD_DIR / "masks_512"

RIGA_LOCAL_IMG_DIR.mkdir(parents=True, exist_ok=True)
RIGA_LOCAL_MASK_DIR.mkdir(parents=True, exist_ok=True)

RIGA_DRIVE_STD_DIR = PATHS["external_datasets"] / "standardized_segmentation" / "RIGA"
RIGA_DRIVE_STD_DIR.mkdir(parents=True, exist_ok=True)

riga_std_rows = []

for idx, row in riga_manifest.reset_index(drop=True).iterrows():
    image_512_path = RIGA_LOCAL_IMG_DIR / f"{row['sample_id']}.jpg"
    mask_512_path = RIGA_LOCAL_MASK_DIR / f"{row['sample_id']}.png"

    original = cv2.imread(row["original_path"], cv2.IMREAD_COLOR)

    if original is None:
        raise RuntimeError(f"Unreadable RIGA original: {row['original_path']}")

    image_512 = resize_to_512(original)

    od_masks = []
    oc_masks = []
    valid_experts = 0

    for expert_id in range(1, 7):
        marked_path = row[f"expert_marking_{expert_id}_path"]
        marked = cv2.imread(marked_path, cv2.IMREAD_COLOR)

        if marked is None:
            raise RuntimeError(f"Unreadable RIGA marking: {marked_path}")

        od_mask, oc_mask, _ = reconstruct_riga_expert_masks(original, marked)

        if od_mask is not None and oc_mask is not None and od_mask.sum() > 0 and oc_mask.sum() > 0:
            od_masks.append(od_mask)
            oc_masks.append(oc_mask)
            valid_experts += 1

    if valid_experts == 0:
        od_consensus = np.zeros((STANDARD_SIZE, STANDARD_SIZE), dtype=np.uint8)
        oc_consensus = np.zeros((STANDARD_SIZE, STANDARD_SIZE), dtype=np.uint8)
        od_threshold = 0
        oc_threshold = 0
    else:
        od_consensus, od_threshold = consensus_mask(od_masks, min_votes=3)
        oc_consensus, oc_threshold = consensus_mask(oc_masks, min_votes=3)
        oc_consensus = (oc_consensus & od_consensus).astype(np.uint8)

    mask_512 = np.zeros((STANDARD_SIZE, STANDARD_SIZE), dtype=np.uint8)
    mask_512[od_consensus == 1] = 1
    mask_512[oc_consensus == 1] = 2

    cv2.imwrite(str(image_512_path), image_512)
    cv2.imwrite(str(mask_512_path), mask_512)

    riga_std_rows.append(
        {
            "dataset": "RIGA",
            "sample_id": row["sample_id"],
            "source": row["source"],
            "subgroup": row["subgroup"],
            "image_512_path": str(image_512_path),
            "mask_512_path": str(mask_512_path),
            "mask_values": "|".join(map(str, sorted(np.unique(mask_512).astype(int).tolist()))),
            "od_pixels_512": int((mask_512 == 1).sum()),
            "oc_pixels_512": int((mask_512 == 2).sum()),
            "valid_experts": int(valid_experts),
            "od_threshold": int(od_threshold),
            "oc_threshold": int(oc_threshold),
        }
    )

    if (idx + 1) % 50 == 0:
        print(f"Processed RIGA: {idx + 1}/{len(riga_manifest)}")

riga_standardized_manifest = pd.DataFrame(riga_std_rows)

riga_standardized_manifest_csv = RIGA_DRIVE_STD_DIR / "riga_standardized_segmentation_manifest.csv"
riga_standardized_manifest_results_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_riga_standardized_segmentation_manifest.csv"

riga_standardized_manifest.to_csv(riga_standardized_manifest_csv, index=False)
riga_standardized_manifest.to_csv(riga_standardized_manifest_results_csv, index=False)

n_images_512 = len(list(RIGA_LOCAL_IMG_DIR.glob("*.jpg")))
n_masks_512 = len(list(RIGA_LOCAL_MASK_DIR.glob("*.png")))
empty_od = int((riga_standardized_manifest["od_pixels_512"] == 0).sum())
empty_oc = int((riga_standardized_manifest["oc_pixels_512"] == 0).sum())
invalid_experts = int((riga_standardized_manifest["valid_experts"] == 0).sum())

print(f"RIGA standardized samples: {len(riga_standardized_manifest)}")
print(f"Local images 512: {n_images_512}")
print(f"Local masks 512: {n_masks_512}")
print(f"Empty OD masks: {empty_od}")
print(f"Empty OC masks: {empty_oc}")
print(f"Samples without valid experts: {invalid_experts}")
print(f"Manifest: {riga_standardized_manifest_csv}")

if len(riga_standardized_manifest) != 750:
    raise RuntimeError("Unexpected RIGA standardized manifest size.")

if n_images_512 != 750 or n_masks_512 != 750:
    raise RuntimeError("Unexpected RIGA standardized local file count.")

if empty_od > 0:
    raise RuntimeError("Empty RIGA OD masks found.")

if empty_oc > 0:
    raise RuntimeError("Empty RIGA OC masks found.")

if invalid_experts > 0:
    raise RuntimeError("RIGA samples without valid expert masks found.")

print("Status: PASS")

In [ ]:
# Cell 24: Audit standardized segmentation datasets

standardized_specs = {
    "REFUGE": {
        "local_root": Path("/content/refuge_standardized_512"),
        "manifest_path": PATHS["external_datasets"] / "standardized_segmentation" / "REFUGE" / "refuge_standardized_segmentation_manifest.csv",
        "expected_samples": 1200,
    },
    "PAPILA": {
        "local_root": Path("/content/papila_standardized_512"),
        "manifest_path": PATHS["external_datasets"] / "standardized_segmentation" / "PAPILA" / "papila_standardized_segmentation_manifest.csv",
        "expected_samples": 488,
    },
    "RIGA": {
        "local_root": Path("/content/riga_standardized_512"),
        "manifest_path": PATHS["external_datasets"] / "standardized_segmentation" / "RIGA" / "riga_standardized_segmentation_manifest.csv",
        "expected_samples": 750,
    },
}

audit_rows = []

for dataset, spec in standardized_specs.items():
    image_dir = spec["local_root"] / "images_512"
    mask_dir = spec["local_root"] / "masks_512"
    manifest_path = spec["manifest_path"]

    manifest_exists = manifest_path.exists()

    if manifest_exists:
        manifest_df = pd.read_csv(manifest_path)
        manifest_rows = len(manifest_df)
        empty_od = int((manifest_df["od_pixels_512"] == 0).sum())
        empty_oc = int((manifest_df["oc_pixels_512"] == 0).sum())
    else:
        manifest_rows = 0
        empty_od = -1
        empty_oc = -1

    local_images = len(list(image_dir.glob("*.jpg"))) if image_dir.exists() else 0
    local_masks = len(list(mask_dir.glob("*.png"))) if mask_dir.exists() else 0

    ready_local = (
        manifest_exists and
        manifest_rows == spec["expected_samples"] and
        local_images == spec["expected_samples"] and
        local_masks == spec["expected_samples"] and
        empty_od == 0 and
        empty_oc == 0
    )

    audit_rows.append(
        {
            "dataset": dataset,
            "manifest_path": str(manifest_path),
            "manifest_exists": manifest_exists,
            "manifest_rows": manifest_rows,
            "expected_samples": spec["expected_samples"],
            "local_images_512": local_images,
            "local_masks_512": local_masks,
            "empty_od_masks": empty_od,
            "empty_oc_masks": empty_oc,
            "ready_local_for_training": ready_local,
        }
    )

standardized_dataset_audit = pd.DataFrame(audit_rows)

standardized_audit_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_standardized_dataset_audit.csv"
standardized_dataset_audit.to_csv(standardized_audit_csv, index=False)

datasets_needing_rebuild = standardized_dataset_audit.loc[
    ~standardized_dataset_audit["ready_local_for_training"],
    "dataset",
].tolist()

print(
    standardized_dataset_audit[
        [
            "dataset",
            "manifest_rows",
            "local_images_512",
            "local_masks_512",
            "empty_od_masks",
            "empty_oc_masks",
            "ready_local_for_training",
        ]
    ].to_string(index=False)
)
print(f"Datasets needing local rebuild: {datasets_needing_rebuild}")
print(f"Audit CSV: {standardized_audit_csv}")

if not standardized_dataset_audit.loc[
    standardized_dataset_audit["dataset"] == "RIGA",
    "ready_local_for_training",
].iloc[0]:
    raise RuntimeError("RIGA is not ready after standardized conversion.")

print("Status: PASS")

In [ ]:
# Cell 25: Build combined standardized segmentation manifest

ready_datasets = standardized_dataset_audit.loc[
    standardized_dataset_audit["ready_local_for_training"],
    "dataset",
].tolist()

combined_rows = []

for _, audit_row in standardized_dataset_audit.iterrows():
    dataset = audit_row["dataset"]

    if not audit_row["ready_local_for_training"]:
        continue

    manifest_df = pd.read_csv(audit_row["manifest_path"]).copy()

    manifest_df["dataset"] = dataset
    manifest_df["standardized_image_exists"] = manifest_df["image_512_path"].apply(lambda x: Path(x).exists())
    manifest_df["standardized_mask_exists"] = manifest_df["mask_512_path"].apply(lambda x: Path(x).exists())
    manifest_df["ready_for_od_training"] = manifest_df["od_pixels_512"] > 0
    manifest_df["ready_for_oc_training"] = manifest_df["oc_pixels_512"] > 0

    combined_rows.append(manifest_df)

combined_standardized_manifest = pd.concat(combined_rows, axis=0).reset_index(drop=True)

required_columns = [
    "dataset",
    "sample_id",
    "image_512_path",
    "mask_512_path",
    "od_pixels_512",
    "oc_pixels_512",
    "standardized_image_exists",
    "standardized_mask_exists",
    "ready_for_od_training",
    "ready_for_oc_training",
]

missing_columns = [
    col for col in required_columns
    if col not in combined_standardized_manifest.columns
]

if missing_columns:
    raise ValueError(f"Missing combined manifest columns: {missing_columns}")

combined_summary = (
    combined_standardized_manifest.groupby("dataset")
    .agg(
        n_samples=("sample_id", "count"),
        images_exist=("standardized_image_exists", "sum"),
        masks_exist=("standardized_mask_exists", "sum"),
        empty_od_masks=("od_pixels_512", lambda x: int((x == 0).sum())),
        empty_oc_masks=("oc_pixels_512", lambda x: int((x == 0).sum())),
        ready_od=("ready_for_od_training", "sum"),
        ready_oc=("ready_for_oc_training", "sum"),
    )
    .reset_index()
)

combined_manifest_csv = (
    PATHS["external_datasets"] /
    "standardized_segmentation" /
    "combined_standardized_segmentation_manifest.csv"
)

combined_manifest_results_csv = (
    PATHS["results_04_od_segmentor_external_validation"] /
    "notebook04_combined_standardized_segmentation_manifest.csv"
)

combined_summary_csv = (
    PATHS["results_04_od_segmentor_external_validation"] /
    "notebook04_combined_standardized_segmentation_summary.csv"
)

combined_standardized_manifest.to_csv(combined_manifest_csv, index=False)
combined_standardized_manifest.to_csv(combined_manifest_results_csv, index=False)
combined_summary.to_csv(combined_summary_csv, index=False)

missing_files = int(
    (~combined_standardized_manifest["standardized_image_exists"]).sum() +
    (~combined_standardized_manifest["standardized_mask_exists"]).sum()
)

empty_od = int((combined_standardized_manifest["od_pixels_512"] == 0).sum())
empty_oc = int((combined_standardized_manifest["oc_pixels_512"] == 0).sum())

duplicate_sample_ids = int(combined_standardized_manifest["sample_id"].duplicated().sum())

print(f"Ready datasets: {ready_datasets}")
print(f"Combined samples: {len(combined_standardized_manifest)}")
print(f"Missing local files: {missing_files}")
print(f"Empty OD masks: {empty_od}")
print(f"Empty OC masks: {empty_oc}")
print(f"Duplicate sample IDs: {duplicate_sample_ids}")
print(f"Combined manifest: {combined_manifest_csv}")

print(combined_summary.to_string(index=False))

if set(ready_datasets) != {"REFUGE", "PAPILA", "RIGA"}:
    raise RuntimeError(f"Unexpected ready datasets: {ready_datasets}")

if len(combined_standardized_manifest) != 2438:
    raise RuntimeError(f"Expected 2438 combined samples, found {len(combined_standardized_manifest)}")

if missing_files > 0:
    raise RuntimeError("Missing local standardized image or mask files found.")

if empty_od > 0 or empty_oc > 0:
    raise RuntimeError("Empty OD or OC masks found in combined manifest.")

if duplicate_sample_ids > 0:
    raise RuntimeError("Duplicate sample IDs found in combined manifest.")

print("Status: PASS")

In [ ]:
# Cell 26: Create dataset-specific segmentation splits

from sklearn.model_selection import train_test_split

SEGMENTATION_SPLIT_SEED = int(PROJECT_CONFIG["seed"])

split_rows = []

# REFUGE: use official train/val/test split
refuge_df = combined_standardized_manifest[
    combined_standardized_manifest["dataset"] == "REFUGE"
].copy()

if "split_original" not in refuge_df.columns:
    raise ValueError("REFUGE split_original column not found.")

for _, row in refuge_df.iterrows():
    split_rows.append(
        {
            "dataset": "REFUGE",
            "sample_id": row["sample_id"],
            "segmentation_split": row["split_original"],
            "split_group": row["sample_id"],
        }
    )

# PAPILA: split by subject ID to avoid OD/OS leakage
papila_df = combined_standardized_manifest[
    combined_standardized_manifest["dataset"] == "PAPILA"
].copy()

papila_df["subject_id"] = papila_df["sample_id"].str.extract(r"(RET[0-9]+)")[0]

if papila_df["subject_id"].isna().any():
    raise RuntimeError("Could not extract all PAPILA subject IDs.")

papila_subjects = papila_df[["subject_id"]].drop_duplicates().reset_index(drop=True)

papila_train_subjects, papila_temp_subjects = train_test_split(
    papila_subjects,
    test_size=0.30,
    random_state=SEGMENTATION_SPLIT_SEED,
    shuffle=True,
)

papila_val_subjects, papila_test_subjects = train_test_split(
    papila_temp_subjects,
    test_size=0.50,
    random_state=SEGMENTATION_SPLIT_SEED,
    shuffle=True,
)

papila_split_map = {
    **{sid: "train" for sid in papila_train_subjects["subject_id"]},
    **{sid: "val" for sid in papila_val_subjects["subject_id"]},
    **{sid: "test" for sid in papila_test_subjects["subject_id"]},
}

for _, row in papila_df.iterrows():
    split_rows.append(
        {
            "dataset": "PAPILA",
            "sample_id": row["sample_id"],
            "segmentation_split": papila_split_map[row["subject_id"]],
            "split_group": row["subject_id"],
        }
    )

# RIGA: split stratified by source/subgroup when possible
riga_df = combined_standardized_manifest[
    combined_standardized_manifest["dataset"] == "RIGA"
].copy()

required_riga_cols = ["source", "subgroup"]

missing_riga_cols = [col for col in required_riga_cols if col not in riga_df.columns]
if missing_riga_cols:
    raise ValueError(f"Missing RIGA columns: {missing_riga_cols}")

riga_train_parts = []
riga_val_parts = []
riga_test_parts = []

for _, group_df in riga_df.groupby(["source", "subgroup"]):
    group_df = group_df.copy()

    train_part, temp_part = train_test_split(
        group_df,
        test_size=0.30,
        random_state=SEGMENTATION_SPLIT_SEED,
        shuffle=True,
    )

    val_part, test_part = train_test_split(
        temp_part,
        test_size=0.50,
        random_state=SEGMENTATION_SPLIT_SEED,
        shuffle=True,
    )

    riga_train_parts.append(train_part)
    riga_val_parts.append(val_part)
    riga_test_parts.append(test_part)

riga_split_df = pd.concat(
    [
        pd.concat(riga_train_parts).assign(segmentation_split="train"),
        pd.concat(riga_val_parts).assign(segmentation_split="val"),
        pd.concat(riga_test_parts).assign(segmentation_split="test"),
    ],
    axis=0,
)

for _, row in riga_split_df.iterrows():
    split_rows.append(
        {
            "dataset": "RIGA",
            "sample_id": row["sample_id"],
            "segmentation_split": row["segmentation_split"],
            "split_group": f"{row['source']}__{row['subgroup']}__{row['sample_id']}",
        }
    )

segmentation_splits = pd.DataFrame(split_rows)

combined_with_splits = combined_standardized_manifest.merge(
    segmentation_splits,
    on=["dataset", "sample_id"],
    how="left",
)

missing_splits = int(combined_with_splits["segmentation_split"].isna().sum())

split_summary = (
    combined_with_splits.groupby(["dataset", "segmentation_split"])
    .agg(
        n_samples=("sample_id", "count"),
        empty_od=("od_pixels_512", lambda x: int((x == 0).sum())),
        empty_oc=("oc_pixels_512", lambda x: int((x == 0).sum())),
    )
    .reset_index()
)

segmentation_splits_csv = (
    PATHS["external_datasets"] /
    "standardized_segmentation" /
    "segmentation_dataset_specific_splits.csv"
)

combined_with_splits_csv = (
    PATHS["external_datasets"] /
    "standardized_segmentation" /
    "combined_standardized_segmentation_manifest_with_splits.csv"
)

split_summary_csv = (
    PATHS["results_04_od_segmentor_external_validation"] /
    "notebook04_segmentation_split_summary.csv"
)

segmentation_splits.to_csv(segmentation_splits_csv, index=False)
combined_with_splits.to_csv(combined_with_splits_csv, index=False)
split_summary.to_csv(split_summary_csv, index=False)

print(f"Split rows: {len(segmentation_splits)}")
print(f"Missing split assignments: {missing_splits}")
print(split_summary.to_string(index=False))
print(f"Manifest with splits: {combined_with_splits_csv}")

if missing_splits > 0:
    raise RuntimeError("Missing segmentation split assignments found.")

if len(combined_with_splits) != 2438:
    raise RuntimeError(f"Expected 2438 rows, found {len(combined_with_splits)}")

print("Status: PASS")

In [ ]:
# Cell 27: Save cross-dataset segmentation experiment plan

segmentation_datasets = ["REFUGE", "PAPILA", "RIGA"]

train_plan_rows = []
eval_plan_rows = []

for train_dataset in segmentation_datasets:
    train_subset = combined_with_splits[
        (combined_with_splits["dataset"] == train_dataset) &
        (combined_with_splits["segmentation_split"] == "train")
    ]

    val_subset = combined_with_splits[
        (combined_with_splits["dataset"] == train_dataset) &
        (combined_with_splits["segmentation_split"] == "val")
    ]

    test_subset = combined_with_splits[
        (combined_with_splits["dataset"] == train_dataset) &
        (combined_with_splits["segmentation_split"] == "test")
    ]

    train_plan_rows.append(
        {
            "segmentor_id": f"S_{train_dataset}",
            "train_dataset": train_dataset,
            "train_samples": len(train_subset),
            "val_samples": len(val_subset),
            "internal_test_samples": len(test_subset),
            "input_size": "512x512",
            "target_classes": "background, optic_disc, optic_cup",
            "primary_seed": int(PROJECT_CONFIG["seed"]),
            "training_role": "dataset_specific_segmentor",
        }
    )

    for test_dataset in segmentation_datasets:
        test_target = combined_with_splits[
            (combined_with_splits["dataset"] == test_dataset) &
            (combined_with_splits["segmentation_split"] == "test")
        ]

        eval_plan_rows.append(
            {
                "segmentor_id": f"S_{train_dataset}",
                "train_dataset": train_dataset,
                "test_dataset": test_dataset,
                "test_samples": len(test_target),
                "evaluation_type": "internal" if train_dataset == test_dataset else "cross_dataset_external",
                "metrics": "Dice_OD,Dice_OC,IoU_OD,IoU_OC,centroid_error_OD,area_error_OD",
            }
        )

segmentation_training_plan = pd.DataFrame(train_plan_rows)
cross_dataset_evaluation_plan = pd.DataFrame(eval_plan_rows)

segmentation_plan_dir = PATHS["external_datasets"] / "standardized_segmentation"
segmentation_training_plan_csv = segmentation_plan_dir / "segmentation_training_plan.csv"
cross_dataset_evaluation_plan_csv = segmentation_plan_dir / "cross_dataset_evaluation_plan.csv"

segmentation_training_plan_results_csv = (
    PATHS["results_04_od_segmentor_external_validation"] /
    "notebook04_segmentation_training_plan.csv"
)

cross_dataset_evaluation_plan_results_csv = (
    PATHS["results_04_od_segmentor_external_validation"] /
    "notebook04_cross_dataset_evaluation_plan.csv"
)

segmentation_training_plan.to_csv(segmentation_training_plan_csv, index=False)
cross_dataset_evaluation_plan.to_csv(cross_dataset_evaluation_plan_csv, index=False)

segmentation_training_plan.to_csv(segmentation_training_plan_results_csv, index=False)
cross_dataset_evaluation_plan.to_csv(cross_dataset_evaluation_plan_results_csv, index=False)

internal_evals = int((cross_dataset_evaluation_plan["evaluation_type"] == "internal").sum())
external_evals = int((cross_dataset_evaluation_plan["evaluation_type"] == "cross_dataset_external").sum())

print(f"Dataset-specific segmentors: {len(segmentation_training_plan)}")
print(f"Evaluation pairs: {len(cross_dataset_evaluation_plan)}")
print(f"Internal evaluations: {internal_evals}")
print(f"Cross-dataset evaluations: {external_evals}")
print(f"Training plan: {segmentation_training_plan_csv}")
print(f"Evaluation plan: {cross_dataset_evaluation_plan_csv}")

if len(segmentation_training_plan) != 3:
    raise RuntimeError("Expected 3 dataset-specific segmentors.")

if len(cross_dataset_evaluation_plan) != 9:
    raise RuntimeError("Expected 9 train-test evaluation pairs.")

if internal_evals != 3 or external_evals != 6:
    raise RuntimeError("Unexpected internal/external evaluation counts.")

print("Status: PASS")

In [ ]:
# Cell 28: Final status

required_outputs = {
    "dataset_registry_csv": dataset_registry_csv,
    "dataset_specs_csv": specs_csv,
    "segmentation_resource_audit_csv": resource_audit_csv,
    "training_plan_csv": training_plan_csv,
    "refuge_manifest_csv": refuge_manifest_csv,
    "papila_manifest_csv": papila_manifest_csv,
    "riga_manifest_csv": riga_manifest_csv,
    "combined_manifest_csv": combined_manifest_csv,
    "combined_with_splits_csv": combined_with_splits_csv,
    "segmentation_training_plan_csv": segmentation_training_plan_csv,
    "cross_dataset_evaluation_plan_csv": cross_dataset_evaluation_plan_csv,
    "standardized_audit_csv": standardized_audit_csv,
}

output_checks = {
    key: path.exists()
    for key, path in required_outputs.items()
}

content_checks = {
    "ready_datasets_are_refuge_papila_riga": set(ready_datasets) == {"REFUGE", "PAPILA", "RIGA"},
    "combined_samples": len(combined_standardized_manifest) == 2438,
    "no_missing_standardized_files": missing_files == 0,
    "no_empty_od_masks": empty_od == 0,
    "no_empty_oc_masks": empty_oc == 0,
    "no_duplicate_sample_ids": duplicate_sample_ids == 0,
    "split_rows": len(segmentation_splits) == 2438,
    "no_missing_splits": missing_splits == 0,
    "segmentors": len(segmentation_training_plan) == 3,
    "evaluation_pairs": len(cross_dataset_evaluation_plan) == 9,
}

final_checks = {**output_checks, **content_checks}
failed_checks = [key for key, value in final_checks.items() if not value]
status = "PASS" if not failed_checks else "FAIL"

notebook04_status = {
    "notebook": "04_external_od_oc_segmentor_training_and_validation",
    "status": status,
    "ready_segmentation_datasets": sorted(list(set(ready_datasets))),
    "pending_segmentation_datasets": ["G1020"],
    "combined_standardized_samples": int(len(combined_standardized_manifest)),
    "refuge_samples": int((combined_standardized_manifest["dataset"] == "REFUGE").sum()),
    "papila_samples": int((combined_standardized_manifest["dataset"] == "PAPILA").sum()),
    "riga_samples": int((combined_standardized_manifest["dataset"] == "RIGA").sum()),
    "dataset_specific_segmentors": int(len(segmentation_training_plan)),
    "cross_dataset_evaluation_pairs": int(len(cross_dataset_evaluation_plan)),
    "internal_evaluations": int(internal_evals),
    "external_evaluations": int(external_evals),
    "failed_checks": failed_checks,
    "next_notebook": "05_train_external_od_oc_segmentors_cross_dataset_validation.ipynb",
}

status_json = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_final_status.json"
status_txt = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_FINAL_STATUS.txt"
status_checks_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook04_final_status_checks.csv"

with open(status_json, "w", encoding="utf-8") as f:
    json.dump(notebook04_status, f, indent=2)

with open(status_txt, "w", encoding="utf-8") as f:
    for key, value in notebook04_status.items():
        f.write(f"{key}: {value}\n")

pd.DataFrame(
    [{"check": key, "passed": bool(value)} for key, value in final_checks.items()]
).to_csv(status_checks_csv, index=False)

print(f"Status: {status}")
print(f"Ready datasets: {sorted(list(set(ready_datasets)))}")
print(f"Combined samples: {len(combined_standardized_manifest)}")
print(f"Segmentors: {len(segmentation_training_plan)}")
print(f"Evaluation pairs: {len(cross_dataset_evaluation_plan)}")
print(f"Final status: {status_json}")

if failed_checks:
    raise RuntimeError(f"Notebook 04 failed checks: {failed_checks}")